# CfC / BAOAB propagator — Anisotropic Gaussian V_θ + Fock-Reg — OpenWebText d=384

## Why this run exists

The `γ=0.10` and `γ=0.30` d=384 runs of
`colab_fock_aniso_gaussian_fockreg_openwebtext.ipynb` both hit chronic
gradient-spike instability under the damped velocity-Verlet integrator.
The `γ=0.10` arm eventually stalled outright: 13 watchdog reloads between
steps 10K and 17K, **zero** PPL improvement over 6,900 steps, and a
record pre-clip gradient of 263,084.

Two structural properties of the Verlet step cause this:

1. **The stiff part of the force is integrated explicitly.** A token in a
   sharp V_θ well has large local curvature `K`; the explicit update is
   stable only while `dt < 2·sqrt(m/K)`. As wells sharpen during training
   the layer step silently crosses that bound and the state amplifies
   geometrically down the remaining layers — which is what a gradient
   spike looks like from outside.
2. **V_θ sits inside the second-order `create_graph` chain**, because the
   force comes from `autograd.grad(V_θ + V_φ, create_graph=True)`.

This notebook runs the fix, in attributable stages, via a single
`INTEGRATOR` switch:

| `INTEGRATOR` | V_θ force | Integrator | What it isolates |
|---|---|---|---|
| `'verlet'` | autograd | damped velocity-Verlet | the existing baseline, bit-identical to the runs above |
| `'analytic_vtheta'` | **closed form** | damped velocity-Verlet | how much of the spiking is the V_θ half of the `create_graph` cascade |
| `'baoab'` | closed form | ABOBA split, exact `exp(-γdt)` friction | the integrator split alone, no CfC |
| `'baoab_cfc'` | closed form | ABOBA split + **closed-form harmonic propagator** for the stiff part of V_θ | the full fix |

Each arm writes to its own Drive folder (the integrator is part of the
variant tag), so arms can be run one at a time and resumed independently.

## What is guaranteed, and by what

- **`'analytic_vtheta'` is the same model, not a different one.**
  `test_cfc_baoab.py::test_analytic_vtheta_equivalence` asserts that
  switching V_θ's force to its closed form leaves the loss and *every
  parameter gradient* unchanged (worst relative error ~2e-6).
- **CfC changes how the force is integrated, not what the force is.**
  The stiff diagonal part of V_θ is propagated by its exact harmonic
  solution and the residual is kicked numerically; the two sum to the
  unmodified total force.
  `test_cfc_baoab.py::test_cfc_force_preservation` verifies the CfC and
  plain-BAOAB steps agree to O(dt³) — a second-order discrepancy would
  mean the force field had changed.
- **The propagator cannot blow up.** All Gaussian wells are attractive, so
  the stiffness is non-negative and the substep is always a bounded
  rotation in phase space. At `K = 10⁴` (ω·dt = 100), twelve explicit
  steps overflow float32 while the CfC step stays inside its initial
  orbit.

Theory: `companion_notes/Closed_Form_and_Hybrid_Integration_Strategies_for_Fock-PARFLM.md`,
`companion_notes/Blended_CfC_BAOAB_Deep_Dive.md`, and `paper_v5` §20.
Implementation: `parf/cfc_baoab.py` (propagator), `parf/model_parf_multixi.py`
(`_layer_step_langevin`), `parf/model_aniso_gaussian_vtheta.py`
(`harmonic_terms`).

## Everything else is held fixed

Same d=384 / L=16 / M=32 architecture, same 5-channel ξ, same anisotropic
depth-conditioned V_θ (5 heads × 8 wells, rank 4), same Fock coupling
regularisation, same WSD schedule, same per-group gradient clips, same
watchdog. Only the integrator changes.


In [ ]:
# == Cell 0: Configuration =============================================

# -- V_theta: Anisotropic Gaussian (diagonal + low-rank precision) -----
V_THETA_VARIANT             = 'aniso_gaussian'
V_THETA_WELLS_PER_HEAD      = 8
V_THETA_DEPTH_CONDITION     = True
V_THETA_DEPTH_CODE_INIT_STD = 0.02
ANISO_RANK                  = 4
W_SCALE                     = 1.0

# -- Xi channels (5long preset from OWT notebook) ----------------------
XI_OVERRIDE     = '5long'
_XI_PRESETS_CFG = {
    5:       [0.25, 0.50, 0.75, 0.95, 0.99],
    '5long': [0.50, 0.75, 0.95, 0.99, 0.995],
    6:       [0.25, 0.50, 0.75, 0.95, 0.99, 0.995],
    '4long': [0.50, 0.75, 0.95, 0.995],
}
XI_ALPHA_INITS = _XI_PRESETS_CFG[XI_OVERRIDE]
XI_CHANNELS    = len(XI_ALPHA_INITS)
V_THETA_N_HEADS = XI_CHANNELS

# -- PARF V_phi --------------------------------------------------------
V_PHI_KIND      = 'structural_competitive'
V_PHI_MLP_HIDDEN = 128
TOP_K           = 16
V_PHI_N_HEADS   = 4
V_PHI_D_TYPE    = 32
V_PHI_D_ANGLE   = 16

# -- Reverse channel stabilisation (E5c) -------------------------------
REVERSE_CHANNEL              = True
REVERSE_CHANNEL_STABLE       = True
REVERSE_CHANNEL_PRE_LN       = True
REVERSE_CHANNEL_SOFT_NORM    = True
REVERSE_CHANNEL_WARMUP_STEPS = 4000
REVERSE_CHANNEL_PER_LAYER    = True
REVERSE_CHANNEL_RESET_SCALE  = False

# -- Register repulsion (B4) -------------------------------------------
REGISTER_REPULSION       = True
REGISTER_REPULSION_COEFF = 0.05
REGISTER_REPULSION_KIND  = 'gram'

# -- Output head -------------------------------------------------------
USE_OUTPUT_BIAS = True
TIE_EMBEDDINGS  = False

# -- Optimizer ---------------------------------------------------------
OPTIMIZER = 'adamw'
GRAD_CENTRALIZATION = False

# -- LR schedule (WSD) -------------------------------------------------
LR_SCHEDULE     = 'wsd'
WSD_WARMUP_FRAC = 0.05
WSD_STABLE_FRAC = 0.60
WSD_LR_FLOOR    = None          # resolved after LR is set

# -- Batch / accumulation ----------------------------------------------
# The CfC arm carries a second anisotropic-well evaluation per layer
# (harmonic_terms at h, plus the force at the drifted h_mid), so its
# activation footprint is ~1.5x the Verlet arm's even with the well
# parameters shared between the two.  The auto-probe in Cell 5 therefore
# tends to land on a smaller per-device batch than the Verlet notebook
# does.  Leaving GRAD_ACCUM fixed would then shrink the *effective*
# batch too, which would confound a Verlet-vs-CfC comparison: the two
# runs would differ in gradient noise as well as in integrator.  So the
# probe compensates -- it keeps EFFECTIVE_BATCH pinned to the target and
# spends the difference on accumulation steps.
#
# 32 matches the Verlet aniso-Gaussian OWT run
# (colab_fock_aniso_gaussian_fockreg_openwebtext.ipynb on an 80GB card:
# batch 16 x accum 2), so PPL curves stay directly comparable.
TARGET_EFFECTIVE_BATCH = 32
GRAD_ACCUM      = 2       # fallback / lower bound; raised by the probe

# -- Fock coupling regularisation --------------------------------------
LAMBDA_FOCK_REG = 5e-3
FOCK_REG_EPS    = 1e-6

# == INTEGRATOR =========================================================
# 'verlet'          : damped velocity-Verlet, friction folded into the
#                     1/(1+dt*gamma) coefficient, V_theta force from
#                     autograd.  The historical baseline -- bit-identical
#                     to the runs this notebook is trying to improve on.
# 'analytic_vtheta' : same integrator, but -grad V_theta comes from its
#                     closed form, so V_theta leaves the second-order
#                     create_graph chain.  Same model, same gradients
#                     (asserted by test_cfc_baoab.py) -- only the way the
#                     force is obtained differs.
# 'baoab'           : palindromic ABOBA split with an exact exp(-gamma*dt)
#                     friction substep and a genuine velocity.
# 'baoab_cfc'       : as 'baoab', plus the closed-form harmonic propagator
#                     for the stiff diagonal part of V_theta.  Immune to
#                     the well-sharpening blow-up that the explicit step
#                     suffers from -- but the anisotropic OFF-diagonal
#                     coupling is still an explicit kick (an omega*dt<2 wall).
# 'baoab_cfc_lowrank': as 'baoab_cfc', but the PSD low-rank part
#                     L = sum_k g_k B_k B_k^T is ALSO integrated exactly, on
#                     its <= n_ctx*K*rank modes (impulse/RESPA fast flow), so
#                     the off-diagonal stiff channel no longer has a hard
#                     omega*dt<2 wall (only narrow damped resonances at
#                     omega*dt ~ k*pi).  Mitigation #1 of the CfC/BAOAB
#                     companion note; pair with PRECISION_LR_MAX (#2).
# INTEGRATOR = 'baoab_cfc'
INTEGRATOR = 'baoab_cfc'


# O-step thermostat temperature.  0.0 = deterministic friction only, which
# keeps this run directly comparable to the Verlet curves.  Raising it
# turns the O-step into a true FDT-locked Langevin thermostat.
LANGEVIN_T = 0.0

_INTEGRATOR_MODES = {
    #                     cfg.integrator        cfg.vtheta_analytic_force
    'verlet':            ('verlet',             False),
    'analytic_vtheta':   ('verlet',             True),
    'baoab':             ('baoab',              True),
    'baoab_cfc':         ('baoab_cfc',          True),
    'baoab_cfc_lowrank': ('baoab_cfc_lowrank',  True),
}
assert INTEGRATOR in _INTEGRATOR_MODES, (
    f'INTEGRATOR={INTEGRATOR!r} not in {sorted(_INTEGRATOR_MODES)}')
CFG_INTEGRATOR, CFG_VTHETA_ANALYTIC = _INTEGRATOR_MODES[INTEGRATOR]

# == Stiffness mitigations (CfC/BAOAB companion note, §29, §41-42) =======
# #2 -- smooth bound on the low-rank curvature sigma_max(B_k)^2 (Frobenius
#       cap).  None keeps B_k unbounded.  Tune against the SCAF Phase 7b/7c
#       Weyl audit ('Weyl frac(>2)' should drop); the Frobenius cap is
#       conservative by up to a factor `rank`, so this is not a literal
#       sigma_max^2 target.  Independent of the integrator.
#
# 2026-09-05 (companion note SS41/SS42): turned ON after
# replay_precision_cap_ablation confirmed budgets of 1.0 AND 4.0 both
# collapse all three captured spikes (step 47116: pre-clip 13,139.5 ->
# 3.98 at budget=1.0; step 48507: 203.1 -> 2.14; step 48917: 202.0 ->
# 2.79) -- including the reverse-channel-led event (48917), which
# neither budget touches directly, implying the two spike "mechanisms"
# (SS41 Findings 2-3) share this one root cause. bracket_precision_lr_max
# (Cell 6b-3) additionally showed the healthy (step 27,000) and
# spike-regime checkpoints have STATISTICALLY SIMILAR ambient
# sigma_max(B_k)^2 under a neutral batch (p50 ~280-310 across all four),
# so there is no tight, tail-only budget available here -- any cap tight
# enough to kill the exponent runaway also compresses everyday operation.
# Chose the more conservative of the two evidenced-safe budgets (1.0 over
# 4.0): cheap to loosen later (`bank._precision_lr_max` is a live,
# hot-swappable Python attribute, not part of state_dict), expensive to
# be wrong (another catastrophic reload) if too loose. Monitor val_ppl /
# dc_ratio / b_proj_sigma_max after resuming and revisit if 1.0 visibly
# hampers learning.
PRECISION_LR_MAX = 1.0
# #1 -- cap on the number of exactly-rotated low-rank modes when
#       INTEGRATOR='baoab_cfc_lowrank' (keeps the stiffest ones).  This is
#       now a genuine cost control: lowrank_modes uses a randomised truncated
#       SVD (torch.svd_lowrank) when this is set, costing O(d*P*q) instead of
#       the full O(d*P^2) per token per layer per step -- the difference
#       between a runnable arm and the batched full-SVD that stalled the run.
#       Only the few stiffest modes cross the omega*dt<2 wall; the rest are
#       demoted to the (stable) explicit kick.  None keeps ALL n_ctx*K*rank
#       (= 5*8*4 = 160 here) modes via the full SVD -- correct but very slow.
LOWRANK_MAX_MODES = 16

# -- Damping coefficient -------------------------------------------------
# gamma=0.100 chosen from the d=384, L=16 aniso-Gaussian+fock-reg gamma
# sweep (colab_fock_gamma_sweep_geodesic_aniso_gaussian_fockreg_d384.ipynb):
# best PPL (278.27) AND best geodesic R_bar (0.6708) coincide at gamma=0.100.
# gamma=0.150/0.250 are tied within ~5% (flat bowl); gamma=0.200 was an
# isolated instability outlier (PPL=2250) bracketed by good neighbours on
# both sides, not a genuine stability wall.
# Under BAOAB the friction is applied as exp(-gamma*dt) rather than
# 1/(1+gamma*dt); at gamma=0.10, dt=1 the two differ by ~0.5%, so the same
# gamma remains directly comparable across arms.
FIXED_GAMMA = 0.10

# -- Regularisation ----------------------------------------------------
LAMBDA_V       = 1e-2
BG_QUAD_EPS    = 0.0

# -- Training ----------------------------------------------------------
TOTAL_STEPS   = 100_000
BLOCK_SIZE    = 512
VOCAB_SIZE    = 50257
SEED          = 0

# -- Depth side-by-side probe (2026-08-23) ------------------------------
# The g0.1 L=16 run hit a burst of large, uncaught grad-clip spikes
# (creation_gate/destruction_gate/register/reverse_ch/depth_code/V_theta,
# steps 6297-6676; see training_log.jsonl around those steps) that a real
# PPL hit (176.88->207.11 across the 6000->6500 eval). depth_code is a
# per-layer nn.Parameter (shape [L, n_ctx, d]); creation_gates/
# destruction_gates are per-layer nn.ModuleLists; reverse_ch is a SINGLE
# module reused (weight-tied) at every layer -- all three mean a smaller
# L shortens the compounding chain a spike has to propagate through both
# forward (activation state) and backward (Jacobian product depth).
# This probe pins d=384 (same as the live run) at a different L, with dt
# left untouched at 1.0 either way (see make_config below), so the ONLY
# thing that differs from the live g0.1/L=16 run is depth itself -- not
# conflated with the separate "fewer L, bigger dt for the same total
# integration time" question, which is a different experiment.
# None -> unchanged default behaviour (ARCH_TIERS ladder in Cell 5 still
# picks d=384, L=16, M=32 first). Set to an int (e.g. 8) to pin that L
# instead; the variant tag below then routes this run to its own Drive
# folder/checkpoints so it cannot collide with the live L=16 run.
L_PROBE_OVERRIDE = 8   # pinned: continuing the 2026-08-23 depth probe

# Stop the training loop cleanly (final eval + checkpoint) once this many
# steps have been taken, without touching TOTAL_STEPS -- the WSD warmup/
# stable/decay windows are fractions of TOTAL_STEPS, so changing
# TOTAL_STEPS itself would also compress/stretch the LR schedule and stop
# this from being an apples-to-apples comparison against the same
# (warmup=5000, stable=5000->65000) schedule phase the L=16 spikes were
# observed in. None = run to TOTAL_STEPS as normal (manual interrupt).
#
# 2026-08-24: the first 8,000-step slice of this probe (stopped by the
# PROBE_MAX_STEPS=8_000 below) ran clean through the entire step window
# (6,297-6,676) where the live L=16 run hit its grad-clip burst -- zero
# [spike] events, monotonically improving PPL (1476.67 -> 136.06). Per
# CfC_BAOAB_Integrator_and_Mitigations.md SS24.4's July-17 precedent
# (Tier-2 "reduce L" delayed but did not prevent the d=768 blowup, which
# hit at step ~=37,000 after 33,000 clean steps), 8,000 clean steps is not
# enough to call this resolved rather than merely delayed -- so this is
# now set to None to let the SAME run (it resumes from the Drive
# checkpoint at step 8,000 automatically; see Cell 2) continue past that
# point and find out when/whether turbulence appears. Re-set to an int to
# stop cleanly again for a snapshot.
PROBE_MAX_STEPS = None   # was 8_000; None = continue past step 8,000

# -- Variant tag (for GDrive path and checkpoint naming) ----------------
_variant_parts = []
_variant_parts.append(f'xi{XI_OVERRIDE}')
_variant_parts.append(f'topk{TOP_K}')
_variant_parts.append(f'dt{V_PHI_D_TYPE}da{V_PHI_D_ANGLE}')
_variant_parts.append(f'mh{V_PHI_N_HEADS}')
_variant_parts.append(f'aniso_dcvt{V_THETA_N_HEADS}x{V_THETA_WELLS_PER_HEAD}')
if L_PROBE_OVERRIDE is not None:
    _variant_parts.append(f'L{L_PROBE_OVERRIDE}probe')
_variant_parts.append('ob')
_variant_parts.append('untied')
_variant_parts.append(LR_SCHEDULE)
_variant_parts.append('e5c')
_variant_parts.append('plgate')
_variant_parts.append(f'rep{REGISTER_REPULSION_COEFF:g}')
_variant_parts.append(f'fockreg{LAMBDA_FOCK_REG:g}')
_variant_parts.append(f'g{FIXED_GAMMA:g}')
# The integrator is part of the tag, so every arm gets its own Drive
# folder, checkpoints and training_log.jsonl and can be resumed on its own.
_variant_parts.append(INTEGRATOR)
if LANGEVIN_T > 0:
    _variant_parts.append(f'T{LANGEVIN_T:g}')
_variant_tag = '_'.join(_variant_parts)

total_wells = V_THETA_N_HEADS * V_THETA_WELLS_PER_HEAD
print(f'Anisotropic Gaussian V_theta on OpenWebText d=384')
print(f'  V_theta: {V_THETA_N_HEADS} heads x {V_THETA_WELLS_PER_HEAD} wells = '
      f'{total_wells} total attractors')
print(f'  Aniso rank r={ANISO_RANK}')
print(f'  Depth-conditioned: {V_THETA_DEPTH_CONDITION}')
print(f'  Fock coupling reg: lambda={LAMBDA_FOCK_REG}  eps={FOCK_REG_EPS}')
print(f'  Damping: fixed_gamma={FIXED_GAMMA}')
print(f'  Integrator: {INTEGRATOR}  '
      f'(cfg.integrator={CFG_INTEGRATOR}, '
      f'analytic_vtheta={CFG_VTHETA_ANALYTIC}, T={LANGEVIN_T:g})')
print(f'  Xi: {XI_CHANNELS}ch  horizons ~{[round(1/(1-a),1) for a in XI_ALPHA_INITS]} tok')
print(f'  V_phi={V_PHI_KIND} x {V_PHI_N_HEADS}h  top_k={TOP_K}  '
      f'd_type={V_PHI_D_TYPE}  d_angle={V_PHI_D_ANGLE}')
print(f'  steps={TOTAL_STEPS}  schedule={LR_SCHEDULE}  grad_accum={GRAD_ACCUM}')
print(f'  [variant] tag={_variant_tag}')

In [ ]:
# == Cell 1: Environment + Drive Mount =================================
import os, sys, gc, shutil, subprocess, json, time, math
from pathlib import Path
from dataclasses import asdict

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

REPO_URL    = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH = 'main'

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'exit {r.returncode}: {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    REPO_ROOT = Path('/content/semsimula-paper')
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}')
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: repo refresh failed ({e}); using existing checkout.')

    _gdrive_name = 'semsimula_fock_cfc_baoab_owt'
    if _variant_tag:
        _gdrive_name += f'_{_variant_tag}'
    GDRIVE_ROOT = Path(f'/content/drive/MyDrive/{_gdrive_name}')
    GDRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    DATA_DIR = GDRIVE_ROOT / 'data'
    DATA_DIR.mkdir(exist_ok=True)
    repo_data = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data.is_symlink():
        repo_data.unlink()
    elif repo_data.is_dir():
        shutil.rmtree(repo_data)
    repo_data.symlink_to(DATA_DIR)

    CKPT_DIR    = GDRIVE_ROOT / 'checkpoints'
    RESULTS_DIR = GDRIVE_ROOT / 'results'
    CKPT_DIR.mkdir(exist_ok=True)
    RESULTS_DIR.mkdir(exist_ok=True)

    _sh('pip install -q transformers huggingface_hub pyarrow matplotlib')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    DATA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    _local_phase = 'cfc_baoab_owt' + (f'_{_variant_tag}' if _variant_tag else '')
    CKPT_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / _local_phase / 'ckpts'
    RESULTS_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / _local_phase
    for d in [DATA_DIR, CKPT_DIR, RESULTS_DIR]:
        d.mkdir(parents=True, exist_ok=True)

CA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
for sub in ['', 'parf', 'multixi', 'scaleup', 'sarf_mass_variant', 'energetic_minima']:
    d = str(CA_DIR / sub) if sub else str(CA_DIR)
    if d not in sys.path:
        sys.path.insert(0, d)

CKPT_PREFIX   = 'fock_cfc_owt' + (f'_{_variant_tag}' if _variant_tag else '')
CKPT_INTERVAL = 7_500
CKPT_STEPS    = list(range(CKPT_INTERVAL, TOTAL_STEPS + 1, CKPT_INTERVAL))

print(f'CKPT_DIR    = {CKPT_DIR}')
print(f'RESULTS_DIR = {RESULTS_DIR}')
print(f'Steps: {TOTAL_STEPS:,}  checkpoints at: {CKPT_STEPS}')

In [ ]:
# == Cell 1b: Resume-path override (same run, different INTEGRATOR) =====
#
# INTEGRATOR is deliberately part of _variant_tag (Cell 0's comment: "so
# every arm gets its own Drive folder ... and can be resumed on its own"),
# which is exactly right when INTEGRATOR is set from the start for a
# genuinely separate side-by-side arm. It is exactly WRONG when the goal
# is to keep training the SAME run and only change which integrator it
# uses from here on (e.g. baoab_cfc -> baoab_cfc_lowrank after a
# hard-watchdog burst): with no override, Cell 1 above already created a
# brand-new, empty GDRIVE_ROOT/CKPT_DIR keyed on the NEW tag, and Cell 2
# below would find no checkpoints there and start from scratch.
#
# Set this to the SOURCE run's variant tag (copy it from that run's own
# Cell 0 printout -- the `[variant] tag=...` line -- or read it off the
# existing Drive folder / checkpoint path) to redirect CKPT_DIR/
# CKPT_PREFIX/RESULTS_DIR/GDRIVE_ROOT there, while the model/training loop
# still use whatever INTEGRATOR is set to in Cell 0. None (default) is a
# no-op: normal per-arm-tag behaviour, unchanged.
#
# 2026-08-27: the live L=8 probe's tag, for switching baoab_cfc ->
# baoab_cfc_lowrank without losing its checkpoint history (see companion
# note SS31 preamble for the run this came from):
# RESUME_VARIANT_TAG_OVERRIDE = None
RESUME_VARIANT_TAG_OVERRIDE = (
     'xi5long_topk16_dt32da16_mh4_aniso_dcvt5x8_L8probe_ob_untied_wsd_'
     'e5c_plgate_rep0.05_fockreg0.005_g0.1_baoab_cfc'
)

if RESUME_VARIANT_TAG_OVERRIDE is not None:
    _old_tag = RESUME_VARIANT_TAG_OVERRIDE
    if IN_COLAB:
        GDRIVE_ROOT = Path(
            f'/content/drive/MyDrive/semsimula_fock_cfc_baoab_owt_{_old_tag}')
        GDRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        CKPT_DIR = GDRIVE_ROOT / 'checkpoints'
        RESULTS_DIR = GDRIVE_ROOT / 'results'
    else:
        _local_phase = f'cfc_baoab_owt_{_old_tag}'
        CKPT_DIR = (REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup'
                     / 'results' / _local_phase / 'ckpts')
        RESULTS_DIR = (REPO_ROOT / 'notebooks' / 'conservative_arch'
                        / 'scaleup' / 'results' / _local_phase)
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    CKPT_PREFIX = f'fock_cfc_owt_{_old_tag}'
    print(f'[resume-override] redirected to source-run tag={_old_tag!r}')
    print(f'  CKPT_DIR    = {CKPT_DIR}')
    print(f'  CKPT_PREFIX = {CKPT_PREFIX}')
    print(f'  Cell 2 below will resume from checkpoints found there; '
          f'training continues under INTEGRATOR={INTEGRATOR!r} '
          f'(cfg.integrator={CFG_INTEGRATOR!r}). New checkpoints / '
          f'best.pt / training_log.jsonl entries land in THIS SAME '
          f'folder from now on, mixing both integrators'' history under '
          f'one filename stream -- copy the folder first if you want a '
          f'clean fork instead of an in-place continuation.')
    print(f'  (the fresh, now-unused GDRIVE_ROOT Cell 1 created for the '
          f'new tag is harmless and can be deleted later.)')


In [ ]:
# == Cell 2: GPU + Checkpoint resolution ===============================
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  ({props.total_memory / 1e9:.1f} GB)')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print('WARNING: No GPU detected. This notebook requires CUDA.')

resume_step = 0
resume_ckpt = None

for s in sorted(CKPT_STEPS, reverse=True):
    cand = CKPT_DIR / f'{CKPT_PREFIX}_step{s}.pt'
    if cand.exists():
        resume_ckpt = cand
        resume_step = s
        break

_best_candidates = []
_canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
if _canonical.exists():
    _best_candidates.append(_canonical)
for _f in sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_best.pt')):
    _best_candidates.append(_f)

_best_path = None
_best_step_found = resume_step
for _cand in _best_candidates:
    try:
        _bd = torch.load(_cand, map_location='cpu', weights_only=False)
        _s = _bd.get('step', 0)
        _p = _bd.get('val_ppl', float('inf'))
        del _bd
        if _s > _best_step_found:
            _best_step_found = _s
            _best_path = _cand
            _best_ppl = _p
            print(f'  Found best candidate: {_cand.name} (step {_s:,}, PPL {_p:.2f})')
    except Exception as e:
        print(f'[warn] could not inspect {_cand.name}: {e}')

if _best_path is not None and _best_step_found > resume_step:
    print(f'Best checkpoint (step {_best_step_found:,}, PPL {_best_ppl:.2f}) is more recent '
          f'than latest periodic checkpoint (step {resume_step:,}) -- resuming from best.')
    resume_ckpt = _best_path
    resume_step = _best_step_found

# Manual checkpoints (save_manual_checkpoint, taken right before a planned
# interrupt) and pre-hard-reload forensic snapshots (_reload_best's
# `_prereload` tag, saved automatically the instant a watchdog hard-trigger
# fires -- these hold the LAST CLEAN weights immediately before the
# offending step, not the corrupted post-update ones) are both full
# (model + optimizer) checkpoints, same as `_probe_stop`. None of the three
# are covered by the periodic/`_best` search above. Left unhandled, a
# session that dies between periodic checkpoints with no new best PPL
# silently falls back to a much older best/periodic checkpoint -- exactly
# the gap that once cost a resume ~20,000 steps it didn't need to lose.
# The step number is parsed straight out of the filename (all three tags
# encode it as `_step<N>_<tag>.pt`) rather than torch.load-ing every
# candidate on disk just to check.
_tagged_re = re.compile(rf'^{re.escape(CKPT_PREFIX)}_step(\d+)_(?:manual|prereload|probe_stop)\.pt$')
_tagged_candidates = []
for _f in CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_*.pt'):
    _m = _tagged_re.match(_f.name)
    if _m:
        _tagged_candidates.append((int(_m.group(1)), _f))
_tagged_candidates.sort()
for _tag_step, _tag_path in _tagged_candidates:
    print(f'  Found manual/prereload/probe_stop candidate: {_tag_path.name} (step {_tag_step:,})')
if _tagged_candidates:
    _best_tag_step, _best_tag_path = _tagged_candidates[-1]
    if _best_tag_step > resume_step:
        print(f'{_best_tag_path.name} (step {_best_tag_step:,}) is more recent than the current '
              f'pick (step {resume_step:,}) -- resuming from it. `_prereload`/`_manual` snapshots '
              f'have no evaluated val_ppl on file; run evaluate() once right after loading if you '
              f'want a fresh reading before committing more training time.')
        resume_ckpt = _best_tag_path
        resume_step = _best_tag_step

# if you want a clean before/after comparison starting from the known-good
# state, force it explicitly rather than trusting the automatic pick — e.g.
# right after Cell 2 runs
#resume_ckpt = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
#resume_step = 27_000  # or read it back from the checkpoint's own 'step' field


if resume_ckpt is not None:
    print(f'Resuming from: {resume_ckpt.name}  (step {resume_step:,})')
    print(f'Remaining: {TOTAL_STEPS - resume_step:,} steps')
else:
    print('No checkpoint found -- training from scratch.')
    print(f'Total: {TOTAL_STEPS:,} steps  Checkpoints every {CKPT_INTERVAL:,}')

In [ ]:
# == Cell 3: Data loading (OpenWebText) ================================
from data_module import get_batch

MAX_TRAIN_TOKENS = 2_000_000_000
VAL_TOKENS       = 2_000_000
CHUNK_SIZE       = 50_000

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('gpt2')

train_cache = DATA_DIR / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
val_cache   = DATA_DIR / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'

for alt_name in [
    'semsimula_fock_depthcond_vtheta_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd_e5c_plgate_rep0.05',
    'semsimula_fock_depthcond_vtheta_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd',
    'semsimula_fock_structured_vtheta_owt_phase4',
    'semsimula_fock_gaussian_sarf_openwebtext_phase5',
    'semsimula_splm_openwebtext_phase4',
    'semsimula_splm_openwebtext_scaleup',
    'semsimula_parf_multixi_openwebtext_scaleup',
    'semsimula_fock_multixi_openwebtext_scaleup',
    'semsimula_splm_openwebtext',
    'semsimula_fock_multihead_openwebtext',
    'semsimula_fock_multicontext_vtheta_owt',
]:
    if train_cache.exists():
        break
    if IN_COLAB:
        alt = Path(f'/content/drive/MyDrive/{alt_name}/data')
    else:
        alt = Path.home() / alt_name / 'data'
    alt_train = alt / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
    alt_val   = alt / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'
    if alt_train.exists():
        import shutil
        print(f'Reusing data cache from {alt}')
        shutil.copy2(str(alt_train), str(train_cache))
        shutil.copy2(str(alt_val), str(val_cache))
        break

if train_cache.exists() and val_cache.exists():
    print('Loading cached OpenWebText tokens ...')
    train_ids = np.load(str(train_cache))
    val_ids   = np.load(str(val_cache))
    print(f'  train: {len(train_ids):,} tokens')
    print(f'  val:   {len(val_ids):,} tokens')
else:
    from datasets import load_dataset
    print(f'Streaming OpenWebText (target: {MAX_TRAIN_TOKENS:,} train + {VAL_TOKENS:,} val tokens) ...')
    ds = load_dataset('Skylion007/openwebtext', split='train', streaming=True, trust_remote_code=True)
    all_ids = []
    total = 0
    target = MAX_TRAIN_TOKENS + VAL_TOKENS
    chunk_texts = []
    n_docs = 0
    t0 = time.time()
    for example in ds:
        chunk_texts.append(example['text'])
        n_docs += 1
        if len(chunk_texts) >= CHUNK_SIZE:
            joined = '\n\n'.join(chunk_texts)
            chunk_ids = tok.encode(joined)
            all_ids.extend(chunk_ids)
            total = len(all_ids)
            elapsed = time.time() - t0
            print(f'  {n_docs:,} docs  {total:,} tokens  ({elapsed:.0f}s)', flush=True)
            chunk_texts = []
            del joined, chunk_ids
            if total >= target:
                break
    if chunk_texts:
        joined = '\n\n'.join(chunk_texts)
        all_ids.extend(tok.encode(joined))
        del joined, chunk_texts
    all_ids = np.array(all_ids, dtype=np.uint16)
    total = len(all_ids)
    print(f'Total streamed: {total:,} tokens from {n_docs:,} documents ({time.time() - t0:.0f}s)')
    val_ids   = all_ids[-VAL_TOKENS:]
    train_ids = all_ids[:-VAL_TOKENS]
    if len(train_ids) > MAX_TRAIN_TOKENS:
        train_ids = train_ids[:MAX_TRAIN_TOKENS]
    del all_ids
    np.save(str(train_cache), train_ids)
    np.save(str(val_cache), val_ids)
    print(f'  Cached: train={len(train_ids):,} -> {train_cache}')
    print(f'  Cached: val={len(val_ids):,}   -> {val_cache}')

print(f'train: {len(train_ids):,}   val: {len(val_ids):,}')

In [ ]:
# == Cell 4: V_theta + integrator: import from the repo, then verify ====
#
# The template notebook inlined AnisotropicMixtureGaussianVTheta.  This one
# imports it instead, because the CfC propagator needs `harmonic_terms`,
# which lives with the class in parf/model_aniso_gaussian_vtheta.py.  An
# inlined copy would silently shadow it and fall back to a stale definition.

from model_aniso_gaussian_vtheta import (
    AnisotropicMixtureGaussianVTheta,
    AnisotropicMultiContextGaussianVTheta,
    AnisotropicDepthConditionedGaussianVTheta,
    install_aniso_depth_routing,
)
import model_parf_multixi as _mpm

# -- Stale-checkout guards (fail here, not 90 minutes into training) -----
assert hasattr(AnisotropicDepthConditionedGaussianVTheta, 'harmonic_terms'), (
    'STALE CHECKOUT: the anisotropic V_theta has no harmonic_terms(), which '
    'the CfC propagator needs. Restart the Colab runtime (Runtime > Restart '
    'runtime) so the freshly fetched module is re-imported.')
assert hasattr(_mpm.MultiXiPARFLM, '_layer_step_langevin'), (
    'STALE CHECKOUT: MultiXiPARFLM has no _layer_step_langevin(). Restart '
    'the Colab runtime and re-run from the top.')
assert 'integrator' in {f.name for f in
                        __import__('dataclasses').fields(_mpm.MultiXiPARFConfig)}, (
    'STALE CHECKOUT: MultiXiPARFConfig has no `integrator` field.')

# -- Run the integrator test suite (CPU, ~10 s) --------------------------
# Cheap insurance: proves on THIS checkout that the analytic V_theta force
# reproduces autograd's gradients exactly, that the CfC split preserves the
# force field to O(dt^3), and that the propagator survives stiffness that
# overflows the explicit step.
import subprocess, sys as _sys
_test = CA_DIR / 'parf' / 'test_cfc_baoab.py'
if _test.exists():
    _res = subprocess.run([_sys.executable, str(_test)],
                          capture_output=True, text=True, cwd=str(CA_DIR / 'parf'))
    print(_res.stdout[-2500:])
    if _res.returncode != 0:
        print(_res.stderr[-2500:])
        raise RuntimeError('CfC/BAOAB integrator tests FAILED -- do not train '
                           'on this checkout.')
else:
    print(f'WARNING: {_test} not found; skipping integrator self-tests.')


In [ ]:
# == Cell 5: Model config + build + aniso V_theta swap =================
import gc
import math
from model_fock_parf_multixi import FockMultiXiPARFLM, FockMultiXiPARFConfig
import model_fock_parf_v2
import model_parf_multixi
import model_parf
import model_parf_sparse

# Guard against a stale in-memory module for the classes THIS RUN
# actually instantiates -- FockMultiXiPARFLM (model_fock_parf_multixi.py)
# on top of MultiXiPARFLM (model_parf_multixi.py). These have their OWN
# copies of the per-layer checkpoint gate and the force-computation
# autograd.grad call; `model_parf.PARFLM` is a *different*, unused base
# class, so checking it (as an earlier version of this cell did) gives a
# false pass. Two independent bugs can each cause the eval-time OOM in
# forward_gathered (all L layers' buffers alive at once during
# evaluate()'s torch.enable_grad() forward, with no outer .backward()
# ever around to free them):
#  1. FockMultiXiPARFLM._stack_forward gating the per-layer checkpoint on
#     `self.training` instead of `torch.is_grad_enabled()`.
#  2. MultiXiPARFLM._layer_step hard-coding `retain_graph=True` on the
#     force autograd.grad call instead of `retain_graph=self.training`
#     (retain_graph is only needed when create_graph=True; in eval it
#     just keeps every layer's buffers alive with no backward() call to
#     ever consume/free them).
# Fail fast here instead of discovering it ~1.5h later at the first eval
# call. If this cell is re-run in a kernel that already imported these
# modules before a later `git fetch/reset`, it will (correctly) still
# fail -- sys.modules caching means only a runtime restart clears it.
import inspect as _inspect
_stack_fwd_src = _inspect.getsource(FockMultiXiPARFLM._stack_forward)
_layer_step_src = _inspect.getsource(model_parf_multixi.MultiXiPARFLM._layer_step)
assert 'torch.is_grad_enabled()' in _stack_fwd_src, (
    'STALE MODULE IN THIS KERNEL: FockMultiXiPARFLM._stack_forward still '
    'gates per-layer checkpointing on `self.training` instead of '
    '`torch.is_grad_enabled()`. Restart the Colab runtime (Runtime > '
    'Restart runtime), re-run the setup cell so it fetches the latest '
    'main, then re-run from the top -- a plain re-run of this cell '
    'cannot fix an already-imported module.'
)
assert 'retain_graph=self.training' in _layer_step_src, (
    'STALE MODULE IN THIS KERNEL: MultiXiPARFLM._layer_step still '
    'hard-codes `retain_graph=True` on the force autograd.grad call '
    '(should be `retain_graph=self.training`). Restart the Colab '
    'runtime and re-run from the top.'
)
print('Eval-time-OOM fix verified present in this kernel '
      '(FockMultiXiPARFLM checkpoint gate + MultiXiPARFLM retain_graph).')

# The BAOAB/CfC integrators return the outgoing velocity through
# _layer_step_ex; a stale model_fock_parf_multixi would still call
# _layer_step and silently drop the O-step, training a Verlet model under a
# BAOAB tag.
_fock_step_src = _inspect.getsource(FockMultiXiPARFLM._fock_layer_step)
assert '_layer_step_ex' in _fock_step_src, (
    'STALE MODULE IN THIS KERNEL: FockMultiXiPARFLM._fock_layer_step still '
    'calls _layer_step instead of _layer_step_ex, so the BAOAB/CfC velocity '
    'would be discarded every layer. Restart the Colab runtime and re-run '
    'from the top.')
print('Integrator plumbing verified (_fock_layer_step -> _layer_step_ex).')

# The V_theta component-sharing fix (context_components) is what keeps
# the CfC arm from building the anisotropic well bank twice per layer.
# A kernel that imported model_aniso_gaussian_vtheta before that commit
# would still OOM at every reasonable batch.
assert hasattr(AnisotropicDepthConditionedGaussianVTheta, 'context_components'), (
    'STALE MODULE IN THIS KERNEL: AnisotropicDepthConditionedGaussianVTheta '
    'has no context_components (the CfC memory fix). Restart the Colab '
    'runtime and re-run from the top.')


LOGFREQ_PATH = CA_DIR / 'scaleup' / 'results' / 'logfreq_surprisal_openwebtext.npy'
DRIVE_LOGFREQ = RESULTS_DIR / 'logfreq_surprisal_openwebtext.npy'

if LOGFREQ_PATH.exists():
    LOGFREQ_FILE = LOGFREQ_PATH
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_FILE = DRIVE_LOGFREQ
else:
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    LOGFREQ_FILE = DRIVE_LOGFREQ
    LOGFREQ_FILE.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_FILE, surprisal)

print(f'Logfreq: {LOGFREQ_FILE}')

if L_PROBE_OVERRIDE is not None:
    # Single, pinned tier: d=384 and M=32 stay identical to the live
    # g0.1/L=16 run so L is the only thing that differs. L=16 already
    # fits at d=384/M=32 on an 80GB card, so a smaller L needs strictly
    # less memory -- no OOM fallback ladder needed here.
    ARCH_TIERS = [(384, L_PROBE_OVERRIDE, 32)]
else:
    ARCH_TIERS = [
        (384, 16, 32),
        (384, 12, 16),
        (256, 16, 16),
        (256,  8, 16),
    ]


def make_config(d, L, n_registers):
    return FockMultiXiPARFConfig(
        vocab_size=VOCAB_SIZE, d=d, max_len=1024,
        L=L, v_hidden=1024, v_depth=3, dt=1.0,
        mass_mode='logfreq',
        logfreq_path=str(LOGFREQ_FILE),
        logfreq_init_alpha=0.1,
        init_gamma=1.0,
        fixed_gamma=FIXED_GAMMA,
        integrator=CFG_INTEGRATOR,
        vtheta_analytic_force=CFG_VTHETA_ANALYTIC,
        lowrank_max_modes=LOWRANK_MAX_MODES,
        langevin_T=LANGEVIN_T,
        causal_force=True,
        ln_after_step=True,
        xi_channels=XI_CHANNELS,
        xi_alpha_inits=XI_ALPHA_INITS,
        xi_learnable=True,
        xi_alpha_init_mode='explicit',
        v_phi_kind=V_PHI_KIND,
        v_phi_d_type=V_PHI_D_TYPE,
        v_phi_d_angle=V_PHI_D_ANGLE,
        v_phi_eps=0.1,
        v_phi_phi_hidden=128,
        v_phi_theta_hidden=128,
        v_phi_mlp_hidden=V_PHI_MLP_HIDDEN,
        top_k=TOP_K,
        v_phi_n_heads=V_PHI_N_HEADS,
        use_output_bias=USE_OUTPUT_BIAS,
        tie_embeddings=TIE_EMBEDDINGS,
        score_head_hidden=32,
        gumbel_tau_init=1.0,
        gumbel_tau_min=0.3,
        gumbel_noise=True,
        use_gathered_v_phi=True,
        use_layer_checkpoint=True,
        ln_before_distance=True,
        per_layer_v_phi_scale=True,
        fock_version='v2',
        n_registers=n_registers,
        register_salience_decay=0.5,
        register_salience_threshold=0.005,
        creation_gate_hidden=64,
        stack_discipline=True,
        d_k=64,
        tau_create_init=8.0,
        reverse_channel=REVERSE_CHANNEL,
        reverse_channel_stable=REVERSE_CHANNEL_STABLE,
        reverse_channel_pre_ln=REVERSE_CHANNEL_PRE_LN,
        reverse_channel_soft_norm=REVERSE_CHANNEL_SOFT_NORM,
        reverse_channel_warmup_steps=REVERSE_CHANNEL_WARMUP_STEPS,
        reverse_channel_per_layer=REVERSE_CHANNEL_PER_LAYER,
        per_register_tau=True,
        per_register_keys=True,
        ortho_register_init=True,
        register_repulsion=REGISTER_REPULSION,
        register_repulsion_coeff=REGISTER_REPULSION_COEFF,
        register_repulsion_kind=REGISTER_REPULSION_KIND,
        prefix_causal_registers=True,
    )


model = None
model_cfg = None
for d, L, M in ARCH_TIERS:
    try:
        cfg = make_config(d, L, M)
        mdl = FockMultiXiPARFLM(cfg).to(DEVICE)
        n_v_theta_mlp = sum(p.numel() for p in mdl.V_theta.parameters())

        _init_log_prec = -math.log(d)
        _prec_max = 2.0 / d
        mdl.V_theta = AnisotropicDepthConditionedGaussianVTheta(
            d=d,
            K=V_THETA_WELLS_PER_HEAD,
            n_ctx=V_THETA_N_HEADS,
            n_layers=cfg.L,
            rank=ANISO_RANK,
            w_scale=W_SCALE,
            init_log_precision=_init_log_prec,
            precision_max=_prec_max,
            precision_lr_max=PRECISION_LR_MAX,
            code_init_std=V_THETA_DEPTH_CODE_INIT_STD,
        ).to(DEVICE)
        install_aniso_depth_routing(mdl)

        n = mdl.num_params()
        n_v_theta = sum(p.numel() for p in mdl.V_theta.parameters())
        print(f'Trying d={d} L={L} M={M} -> {n:,} params '
              f'(V_theta {n_v_theta_mlp:,} MLP -> {n_v_theta:,} Aniso-Gaussian)')

        if DEVICE == 'cuda':
            def _arch_probe(_m):
                rng = np.random.default_rng(42)
                xb, yb = get_batch(train_ids, 2, BLOCK_SIZE, rng)
                x = torch.from_numpy(xb).to(DEVICE)
                y = torch.from_numpy(yb).to(DEVICE)
                _, loss = _m(x, y)
                loss.backward()
                _m.zero_grad(set_to_none=True)
                del x, y, xb, yb, loss
            _arch_probe(mdl)
            gc.collect()
            torch.cuda.empty_cache()
            print(f'OOM probe passed (batch=2)')
        model = mdl
        model_cfg = cfg
        break
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            print(f'  OOM at d={d} L={L} M={M} -- trying next tier ...')
            del mdl
            gc.collect()
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
            continue
        raise

if model is None:
    raise RuntimeError('All architecture tiers OOMed.')

if USE_OUTPUT_BIAS:
    _ob_counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE)
    model.init_output_bias_from_logfreq(_ob_counts)
    print(f'Output bias <- log-unigram-freq  '
          f'(b range [{model.out_bias.min().item():.2f}, '
          f'{model.out_bias.max().item():.2f}])')

# -- Auto batch size ----------------------------------------------------
# Two measured facts this probe has to respect:
#
# 1. CUDA OOM is not recoverable in-process (60+ GB leftover after
#    traceback-clear + empty_cache).  Never start a forward that we
#    already have reason to believe will OOM.
# 2. `total_memory / 1e9` is ~85 on an 80 GiB card (decimal GB vs GiB).
#    A 92% budget of 85 GB is 78 GB -- essentially the whole device --
#    which is how batch=4 passed the probe and then died on step 1
#    at 78.6 GiB allocated.  Budget is 80% of total_memory, in bytes.
#
# Procedure: measure a train-shaped step at bs=2 (the architecture
# probe already survived this, so it will not OOM).  Larger sizes are
# attempted only when a linear-in-batch estimate of their peak plus
# the AdamW reserve sits under the 80% budget.  On an 80 GiB card the
# CfC arm's bs=2 peak is already ~half the device, so bs=4 is skipped
# and we train at 2 x 16 = 32 -- same effective batch as Verlet.
import gc

_n_params = sum(p.numel() for p in model.parameters())
_reserve_bytes = 2.5 * _n_params * 4          # AdamW exp_avg+exp_avg_sq + slack
_reserve = _reserve_bytes / 1e9


def _clear_exc():
    for _a in ('last_traceback', 'last_value', 'last_type', 'last_exc'):
        if hasattr(sys, _a):
            setattr(sys, _a, None)
    try:
        ip = get_ipython()
        if ip is not None:
            for _a in ('_last_traceback', 'last_execution_result'):
                if hasattr(ip, _a):
                    setattr(ip, _a, None)
    except NameError:
        pass


def _release():
    _clear_exc()
    model.zero_grad(set_to_none=True)
    gc.collect()
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def _rebuild_model():
    """Drop the live module and reconstruct it.  Weights are still
    random at this point (the training cell loads any checkpoint), so
    this does not discard trained state."""
    global model
    d, L, M = model_cfg.d, model_cfg.L, model_cfg.n_registers
    _clear_exc()
    try:
        model.zero_grad(set_to_none=True)
    except Exception:
        pass
    del model
    gc.collect()
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    cfg = make_config(d, L, M)
    mdl = FockMultiXiPARFLM(cfg).to(DEVICE)
    _init_log_prec = -math.log(d)
    _prec_max = 2.0 / d
    mdl.V_theta = AnisotropicDepthConditionedGaussianVTheta(
        d=d, K=V_THETA_WELLS_PER_HEAD, n_ctx=V_THETA_N_HEADS,
        n_layers=cfg.L, rank=ANISO_RANK, w_scale=W_SCALE,
        init_log_precision=_init_log_prec, precision_max=_prec_max,
        precision_lr_max=PRECISION_LR_MAX,
        code_init_std=V_THETA_DEPTH_CODE_INIT_STD,
    ).to(DEVICE)
    install_aniso_depth_routing(mdl)
    if USE_OUTPUT_BIAS:
        mdl.init_output_bias_from_logfreq(_ob_counts)
    model = mdl
    _held = torch.cuda.memory_allocated() / 1e9 if DEVICE == 'cuda' else 0.0
    print(f'  rebuilt model  (allocated {_held:.1f} GB)')
    return _held


def _probe_batch(bs):
    """One train-shaped step: stack + V_theta regularizer + backward.

    The training cell's forward_with_vreg is this plus the (tiny) Fock
    coupling term.  The extra V_theta call is the piece the previous
    probe omitted and that shows up as a step-1 OOM after a 'passing'
    probe.
    """
    rng = np.random.default_rng(42)
    xb, yb = get_batch(train_ids, bs, BLOCK_SIZE, rng)
    x = torch.from_numpy(xb).to(DEVICE)
    y = torch.from_numpy(yb).to(DEVICE)
    h0 = model._embed(x)
    h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
    logits = model.compute_logits(h_L)
    loss = F.cross_entropy(
        logits.reshape(-1, model_cfg.vocab_size), y.reshape(-1),
    )
    if LAMBDA_V > 0:
        xis = model.xi_module(h_L.detach())
        V_vals = model.V_theta(xis, h_L)
        loss = loss + LAMBDA_V * (V_vals.float() ** 2).mean()
    loss.backward()
    peak = torch.cuda.max_memory_allocated() / 1e9
    del x, y, xb, yb, loss, h0, h_L, logits
    return peak


BATCH_SIZE = 2
if DEVICE == 'cuda':
    _vram_bytes = torch.cuda.get_device_properties(0).total_memory
    _vram_gb = _vram_bytes / 1e9
    _budget_gb = 0.80 * _vram_gb
    _release()
    _base_gb = torch.cuda.memory_allocated() / 1e9
    print(f'Batch probe on {_vram_gb:.1f} GB device '
          f'(budget {_budget_gb:.1f} GB = 80%): '
          f'weights {_base_gb:.1f} GB, AdamW reserve {_reserve:.1f} GB')

    # Measure the known-good size.  Do not skip this: the architecture
    # probe's batch=2 was a bare model() call, not the training step.
    _peak2 = _probe_batch(2)
    _release()
    print(f'  bs= 2: peak {_peak2:.1f} GB')
    if _peak2 + _reserve > _budget_gb:
        print(f'    even bs=2 is over budget; training at 2 anyway '
              f'(accum will be {TARGET_EFFECTIVE_BATCH // 2})')
        BATCH_SIZE = 2
    else:
        BATCH_SIZE = 2
        _act2 = max(_peak2 - _base_gb, 0.0)
        for bs in (4, 8, 16):
            _est = _base_gb + _act2 * (bs / 2.0)
            if _est + _reserve > _budget_gb:
                print(f'  bs={bs:>2}: estimated {_est:.1f} GB + reserve '
                      f'exceeds {_budget_gb:.1f} GB -- skipping '
                      f'(not attempting; an OOM here is unrecoverable)')
                break
            _peak = None
            try:
                _peak = _probe_batch(bs)
            except RuntimeError as _err:
                _is_oom = 'out of memory' in str(_err).lower()
                _err.__traceback__ = None
                _err.__context__ = None
                _err.__cause__ = None
                del _err
                if not _is_oom:
                    raise
                print(f'  bs={bs:>2}: OOM despite estimate {_est:.1f} GB '
                      f'-- keeping bs={BATCH_SIZE}')
                _rebuild_model()
                break
            _release()
            if _peak + _reserve > _budget_gb:
                print(f'  bs={bs:>2}: peak {_peak:.1f} GB + reserve '
                      f'exceeds budget -- keeping bs={BATCH_SIZE}')
                break
            BATCH_SIZE = bs
            print(f'  bs={bs:>2}: peak {_peak:.1f} GB  OK')
            _act2 = max(_peak - _base_gb, 0.0) / (bs / 2.0)

GRAD_ACCUM = max(1, TARGET_EFFECTIVE_BATCH // BATCH_SIZE)
EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
print(f'Auto batch: {BATCH_SIZE} x accum={GRAD_ACCUM} '
      f'(eff={EFFECTIVE_BATCH}, target={TARGET_EFFECTIVE_BATCH})')
if EFFECTIVE_BATCH != TARGET_EFFECTIVE_BATCH:
    print(f'  NOTE: effective batch {EFFECTIVE_BATCH} != target '
          f'{TARGET_EFFECTIVE_BATCH}; PPL is not directly comparable to '
          f'the Verlet run at eff=32.')
n_params = model.num_params()
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())

d = model_cfg.d
print(f'\nModel: FockMultiXiPARFLM v2.1 + Anisotropic Gaussian V_theta')
print(f'  params: {n_params:,}  (V_theta: {n_v_theta:,})')
print(f'  d={d}  L={model_cfg.L}  M={model_cfg.n_registers}')
print(f'  V_theta: aniso-gaussian  rank={ANISO_RANK}  '
      f'{V_THETA_N_HEADS} heads x {V_THETA_WELLS_PER_HEAD} wells = '
      f'{V_THETA_N_HEADS * V_THETA_WELLS_PER_HEAD} attractors')
print(f'  fock-reg: lambda={LAMBDA_FOCK_REG}  eps={FOCK_REG_EPS}')
print(f'  integrator: {INTEGRATOR}  gamma={FIXED_GAMMA}  T={LANGEVIN_T:g}')
print(f'  V_phi={V_PHI_KIND} x {V_PHI_N_HEADS} head(s)  top_k={TOP_K}  '
      f'd_type={V_PHI_D_TYPE}  d_angle={V_PHI_D_ANGLE}')
print(f'  batch={BATCH_SIZE} x accum={GRAD_ACCUM} (eff={EFFECTIVE_BATCH})')

In [ ]:
# == Cell 6d: Gradient-spike replay -- Phase 2 op/layer forensics ==========
# Root-cause diagnostic Phase 2 (CfC/BAOAB companion note SS33.3): given a
# Phase 1 '_spikebatch.pt' bundle (pre-optim.step() weights + the exact
# microbatch(es) + RNG state, captured at a GRAD_NORM_HARD_TRIGGER event),
# deterministically replay that one step's forward+backward in isolation,
# instrumented with:
#   (a) per-parameter grad norms -- finer than per-group, e.g.
#       creation_gate_qkv.log_tau vs. W_Q, or a single reverse_channel_scale.
#   (b) per-layer grad attribution -- a hook on the h tensor at each of the
#       L layer boundaries (gradient flows L -> 0; a localised spike is a
#       strong signal, and dovetails with SS27's boundary-layer finding).
#   (c) forward-activation extremes at the numerically risky ops the
#       companion note names: creation-gate softmax temperature tau,
#       cumulative-softmax denominator Z (via its alpha_max proxy),
#       reverse-channel logit_scale and the Q_force it injects, the
#       destruction-gate output, and the V_theta quadratic-form exponent
#       (recomputed from the bank's own public _components()) -- now
#       layer-resolved for creation_gate_qkv/reverse_ch (2026-08-30,
#       companion note SS38 follow-up): both are single shared modules
#       called once per layer, and _record() used to overwrite the same
#       dict key on every call, so activation_extremes silently only ever
#       reflected the LAST layer. V_theta banks turned out NOT to have
#       this bug at all once inspected against real replay output: they
#       are called ONCE per forward pass on the full depth-stacked xi
#       tensor, not per _fock_layer_step, so they're reported at the
#       fixed sentinel layer=-1 (not layer-attributable), same convention
#       as reverse_ch.logit_scale.
#   (d) the official per-layer register/gate health buffer
#       (model.set_fock_capture(True) -> model._fock_capture), giving
#       active_frac / salience / reg_cos_sim / destroy_mean /
#       create_alpha_max / create_entropy / rev_entropy / rev_alpha_max /
#       rev_scale / qforce_ratio FOR EVERY LAYER for free.
#   (e2) per-bank V_theta exponent OCCUPANCY HISTOGRAM (SS39) -- how many
#       token slots sit in the band where exp(exponent) is still alive
#       (> -10) versus underflowed to zero, since min/max/mean provably
#       could not separate the two modes but density is what the
#       localized-mode conjecture is actually about.
#   (f) attribute_spike_rows(step_tag): re-runs the captured batch one row
#       at a time to test whether a small minority of sequences owns the
#       depth_code gradient direction (SS39) -- the question that matters
#       once you notice depth_code is clipped to the same 0.25 on every
#       step regardless of how big its pre-clip norm was.
#   (e) inspect_spike_tokens(step_tag): decodes the exact offending
#       microbatch's GPT-2 tokens (via Cell 3's `tok`) and ranks rows by
#       longest same-token repeat run, to check whether a degenerate
#       OpenWebText passage correlates with the localized-blowup mode
#       (SS38.2) -- pure CPU bookkeeping, no model/GPU/RNG state touched.
#
# Non-pollution invariant: this cell saves the live model's weights,
# .grad tensors, and RNG state before disturbing any of them, and restores
# all three when done -- resuming the training loop afterward is safe
# (mirrors the SCAF GradientSpikeProbe design's save/zero/restore
# contract, docs/Gradient_Spike_Probe_Requirements_and_Design.md SS7 in
# semsimula-scaf).
#
# CAVEAT: (a) and (b) are exact -- they only touch documented, already-used
# training-loop machinery. (c)'s hooks are best-effort: they were derived
# from reading model_fock_parf_v2.py / model_fock_parf_multixi.py /
# model_aniso_gaussian_vtheta.py, not executed against the live model, so a
# `[replay][WARN] could not instrument ...` line for any one of them is
# possible and does not invalidate the others -- report which ones fire so
# they can be patched up.
#
# Usage: this cell only *defines* functions, so it is safe (and, since
# 2026-08-30 companion note SS37, intended) to run it right after Cell 5
# builds `model`, before Cell 6 starts the long-running training loop --
# that's the whole point of interrupt -> replay_all_captures() ->
# run_training(next_step, TOTAL_STEPS) needing nothing predefined elsewhere.
# The functions below still *call* into a handful of Cell-6 globals
# (forward_with_vreg, LAMBDA_V/LAMBDA_FOCK_REG/FOCK_REG_EPS, _GRAD_CLIP_CFG,
# WATCHDOG_EXCLUDE_GROUPS, CKPT_DIR/CKPT_PREFIX) looked up at call time, so
# Cell 6 must have at least started (its config/setup lines run near-
# instantly, well before the loop itself blocks) before you actually call
# replay_spike_batch(...) / replay_all_captures() -- just not before this
# cell defines them.
#   replay_spike_batch(34091)
#   replay_spike_batch(32139)
#   replay_all_captures()

import copy as _copy
from grad_clip_utils import assign_clip_group, per_group_grad_norms


def _isolated_grad_snapshot(mdl):
    """Save existing .grad tensors (or None) for every parameter."""
    return {n: (p.grad.detach().clone() if p.grad is not None else None)
            for n, p in mdl.named_parameters()}


def _isolated_grad_restore(mdl, saved):
    for n, p in mdl.named_parameters():
        g = saved.get(n)
        p.grad = g.clone() if g is not None else None


def replay_spike_batch(step_tag, mdl=None, top_k=12, verbose=True):
    """Deterministically replay the forward+backward captured by Phase 1
    at a GRAD_NORM_HARD_TRIGGER event, with op/layer-resolved forensics.

    step_tag : int or str -- the step number in the
        '..._step{tag}_spikebatch.pt' filename (e.g. 34091).
    mdl : the live `model` by default.

    Returns a dict report; also prints a human-readable summary. Restores
    mdl's weights/.grad/RNG state to what they were before the call.
    """
    mdl = mdl if mdl is not None else model
    path = CKPT_DIR / f'{CKPT_PREFIX}_step{step_tag}_spikebatch.pt'
    # map_location='cpu' (not DEVICE): rng_state_cpu/rng_state_cuda must stay
    # plain CPU ByteTensors for torch.(cuda.)set_rng_state -- moving them to
    # DEVICE here breaks that ("RNG state must be a torch.ByteTensor"). The
    # model_state_dict tensors don't need this either: they're already
    # explicitly re-mapped with `.to(DEVICE)` per-tensor below.
    bundle = torch.load(path, map_location='cpu', weights_only=False)
    print(f'[replay] loaded {path.name}  step={bundle["step"]}  '
          f'pre_clip_grad_norm={bundle["pre_clip_grad_norm"]}')
    if bundle.get('top_groups'):
        print('[replay] Phase-0 top groups at capture time: '
              + ', '.join(f'{k}={v}' for k, v in bundle['top_groups'].items()))

    # -- non-pollution: snapshot everything this call is about to disturb --
    _saved_grads = _isolated_grad_snapshot(mdl)
    _saved_sd = _copy.deepcopy(mdl.state_dict())
    _saved_rng_cpu = torch.get_rng_state()
    _saved_rng_cuda = torch.cuda.get_rng_state_all() if DEVICE == 'cuda' else None
    _restore_fns = []

    # -- (b) per-layer instrumentation: wrap whatever `_fock_layer_step` is
    #    CURRENTLY bound to (e.g. the aniso depth-routing wrapper installed
    #    by install_aniso_depth_routing), so this composes with it rather
    #    than clobbering it. Caveat: if USE_LAYER_CHECKPOINT is on, gradient
    #    checkpointing recomputes the forward during backward, so a hook
    #    may fire twice for the same layer -- the second firing overwrites
    #    the first, which is fine for peak-magnitude attribution. --
    _layer_grad_norms = {}

    def _make_layer_hook(ell):
        def _hook(grad):
            _layer_grad_norms[ell] = float(grad.detach().norm())
        return _hook

    # 2026-08-30 (companion note SS38 follow-up): creation_gate_qkv,
    # reverse_ch, and each V_theta bank are SINGLE shared modules called
    # once per _fock_layer_step invocation -- i.e. once per layer, L times
    # per forward pass -- unlike destruction_gates (a per-layer ModuleList,
    # so destruction_gates[idx] already disambiguates by construction).
    # Before this fix, _record()'s hooks on those shared modules simply
    # overwrote the same dict key on every layer's call, so
    # activation_extremes silently only ever reflected the LAST layer (7)
    # -- exactly the layers NOT implicated in the localized-blowup mode
    # (layers 0-2). _current_layer is set by the wrapper below just before
    # calling into the real layer step, so _record can tag every reading
    # with the layer that actually produced it.
    _current_layer = [-1]

    _orig_layer_step = mdl._fock_layer_step

    def _instrumented_layer_step(h, h_prev, r, salience, m_b, gamma, dt,
                                  layer_idx, *args, **kwargs):
        _current_layer[0] = layer_idx
        out = _orig_layer_step(h, h_prev, r, salience, m_b, gamma, dt,
                                layer_idx, *args, **kwargs)
        h_new = out[0]
        if torch.is_tensor(h_new) and h_new.requires_grad:
            h_new.register_hook(_make_layer_hook(layer_idx))
        return out

    mdl._fock_layer_step = _instrumented_layer_step
    _restore_fns.append(lambda: setattr(mdl, '_fock_layer_step', _orig_layer_step))

    # -- (c) forward-activation-extreme instrumentation (best-effort),
    #    now layer-resolved: _act_extremes[name] is {layer_idx: stats},
    #    not a single overwritten dict --
    _act_extremes = {}

    def _record(name, t, layer=None):
        # layer=None (default): tag with whatever layer is CURRENTLY
        # executing, per _current_layer above -- correct for hooks that
        # fire from inside _fock_layer_step (creation_gate_qkv, reverse_ch).
        # layer=-1 (explicit): this op is NOT layer-scoped at all -- pass
        # it explicitly rather than trusting _current_layer, which would
        # otherwise silently report whatever layer happened to run last
        # (companion note SS38 follow-up #2: V_theta banks are called
        # ONCE per forward pass on the full depth-stacked xi tensor, not
        # per _fock_layer_step -- their hook fires AFTER the layer loop
        # finishes, so _current_layer[0] would read 7, the last layer
        # processed, which is not what "layer" means for this op at all).
        if layer is None:
            layer = _current_layer[0]
        try:
            tf = t.detach().float()
            stats = {
                'min': float(tf.min()), 'max': float(tf.max()),
                'mean': float(tf.mean()),
            }
        except Exception as _e:
            stats = {'error': str(_e)}
        _act_extremes.setdefault(name, {})[layer] = stats

    # -- (c2) V_theta exponent HISTOGRAM (companion note SS39): min/max/mean
    #    cannot answer the question the localized-mode conjecture actually
    #    poses, which is one of DENSITY -- how many tokens sit in the narrow
    #    band where exp(exponent) is still numerically alive (> ~-10, i.e.
    #    exp() >= 4.5e-5) and the well therefore contributes real gradient,
    #    versus the overwhelming majority that have underflowed to exactly
    #    zero and contribute none at all. The max over ~16k tokens is a
    #    single order statistic, and it actually went the "wrong" way
    #    between the two modes (bank[3] max was -1.76/-1.73 on the two
    #    smooth captures vs. -2.48/-3.91 on the two localized ones), so
    #    per-bank occupancy counts are what's needed instead. --
    # Top edge is +inf, not 0, even though exponent = -0.5*(nonneg) can only
    # be <= 0 mathematically: at diff==0 it can come back a few 1e-12 ABOVE
    # zero in float32, and a bin list topped at 0.0 silently drops exactly
    # those slots -- which are live-band tokens, so the drop would understate
    # live_frac, the one number this histogram exists to measure. With +inf
    # the bins cover the whole real line and sum(counts) == numel always.
    _EXP_EDGES = [float('inf'), -1.0, -2.0, -5.0, -10.0, -20.0, -30.0,
                  -100.0, -1e3, -1e4, float('-inf')]
    _EXP_LIVE_BINS = 4          # bins spanning (-10, +inf]
    _exp_hist = {}

    def _accum_exponent_hist(idx, exponent):
        try:
            with torch.no_grad():
                e = exponent.detach().float().reshape(-1)
                counts = _exp_hist.setdefault(idx, [0] * (len(_EXP_EDGES) - 1))
                for b in range(len(_EXP_EDGES) - 1):
                    hi, lo = _EXP_EDGES[b], _EXP_EDGES[b + 1]
                    counts[b] += int(((e <= hi) & (e > lo)).sum())
        except Exception:
            pass

    # creation-gate temperature + cumulative-softmax Z (forward_prefix is
    # called directly, not via __call__, so a plain forward hook would
    # never fire -- monkeypatch the bound method instead).
    try:
        _cg = mdl.creation_gate_qkv
        _orig_fp = _cg.forward_prefix

        def _wrapped_fp(h, r):
            readout, alpha_max = _orig_fp(h, r)
            _record('creation_gate.alpha_max (~Z-normalised top weight)', alpha_max)
            if getattr(_cg, 'log_tau', None) is not None:
                _record('creation_gate.tau', _cg.log_tau.exp().clamp(min=1e-4))
            return readout, alpha_max

        _cg.forward_prefix = _wrapped_fp
        _restore_fns.append(lambda: setattr(_cg, 'forward_prefix', _orig_fp))
    except Exception as _e:
        print(f'[replay][WARN] could not instrument creation_gate_qkv: {_e}')

    # reverse channel: logit_scale is a static parameter (read directly,
    # no hook needed); Q_force is the batch-dependent activation the
    # companion note flags as the non-conservative injection -- hook the
    # module call directly since `self.reverse_ch(...)` goes via __call__.
    try:
        _rc = mdl.reverse_ch
        if getattr(_rc, 'logit_scale', None) is not None:
            _record('reverse_ch.logit_scale (exp, clamped)',
                     _rc.logit_scale.exp().clamp(
                         max=getattr(_rc, 'logit_scale_max', 100.0)))

        def _rc_fwd_hook(module, inputs, output):
            _record('reverse_ch.Q_force', output)

        _h_rc = _rc.register_forward_hook(_rc_fwd_hook)
        _restore_fns.append(_h_rc.remove)
    except Exception as _e:
        print(f'[replay][WARN] could not instrument reverse_ch: {_e}')

    # V_theta: exponent / diag-vs-low-rank split and the depth-code-shifted
    # xi magnitude, recomputed from each per-channel bank's own public
    # _components() using the (xi, h) this forward call actually saw.
    try:
        _banks = mdl.V_theta.banks

        def _make_vtheta_hook(idx):
            def _vt_fwd_hook(module, args, kwargs, output):
                xi_in = args[0] if len(args) > 0 else kwargs.get('xis', kwargs.get('xi'))
                h_in = args[1] if len(args) > 1 else kwargs.get('h')
                try:
                    mu, a, w, B = module._components(xi_in)
                    h_e = h_in.unsqueeze(-2)
                    diff = h_e - mu
                    diag_term = (a * diff * diff).sum(dim=-1)
                    if B.shape[-1] > 0:
                        Bt_diff = torch.einsum('...kd,...kdr->...kr', diff, B)
                        lr_term = (Bt_diff * Bt_diff).sum(dim=-1)
                    else:
                        lr_term = torch.zeros_like(diag_term)
                    exponent = -0.5 * (diag_term + lr_term)
                    _accum_exponent_hist(idx, exponent)
                    _record(f'V_theta.bank[{idx}].exponent', exponent, layer=-1)
                    _record(f'V_theta.bank[{idx}].lr_term_share',
                             lr_term / (diag_term + lr_term).clamp(min=1e-12),
                             layer=-1)
                    _record(f'V_theta.bank[{idx}].xi_shifted_norm',
                             xi_in.detach().float().norm(dim=-1), layer=-1)
                except Exception as _e2:
                    _act_extremes[f'V_theta.bank[{idx}]'] = {'error': str(_e2)}
            return _vt_fwd_hook

        for _idx, _bank in enumerate(_banks):
            _h_vt = _bank.register_forward_hook(_make_vtheta_hook(_idx), with_kwargs=True)
            _restore_fns.append(_h_vt.remove)
    except Exception as _e:
        print(f'[replay][WARN] could not instrument V_theta banks: {_e}')

    # register salience gate: capture what the destruction gates emit --
    # salience itself is thresholded at cfg.register_salience_threshold
    # (0.005) upstream of this call; the gate's own output extremes are
    # the batch-dependent signal available without touching model source.
    try:
        _dgs = mdl.destruction_gates

        def _make_dg_hook(idx):
            def _dg_fwd_hook(module, inputs, output):
                _record(f'destruction_gates[{idx}].output', output)
            return _dg_fwd_hook

        for _idx, _dg in enumerate(_dgs):
            _h_dg = _dg.register_forward_hook(_make_dg_hook(_idx))
            _restore_fns.append(_h_dg.remove)
    except Exception as _e:
        print(f'[replay][WARN] could not instrument destruction_gates: {_e}')

    # -- official per-layer diagnostic buffer (companion note SS38 follow-up):
    #    FockMultiXiPARFLM.set_fock_capture(True) makes _fock_layer_step
    #    append a @torch.no_grad() dict of register/gate health scalars
    #    (active_frac, salience_mean/std, reg_cos_sim, destroy_mean,
    #    qforce_ratio, rev_scale, create_alpha_max, create_entropy,
    #    rev_entropy, rev_alpha_max) FOR EVERY LAYER on every forward --
    #    this is the same API the eval-time causal/register diagnostics use
    #    (model.eval(); model.set_fock_capture(True)), just enabled here
    #    during the replayed training-mode forward+backward instead. Being
    #    torch.no_grad() internally, it costs almost nothing and cannot
    #    perturb the backward pass it runs alongside.
    if not hasattr(mdl, 'set_fock_capture'):
        print('[replay][WARN] model has no set_fock_capture(); skipping '
              'per-layer register/gate diagnostic buffer.')
    else:
        mdl.set_fock_capture(True)
        _restore_fns.append(lambda: mdl.set_fock_capture(False))
    _fock_capture_per_microbatch = []

    try:
        # -- load the pinned pre-step weights + RNG state --
        mdl.load_state_dict(
            {k: v.to(DEVICE) for k, v in bundle['model_state_dict'].items()},
            strict=False)
        torch.set_rng_state(bundle['rng_state_cpu'])
        if bundle.get('rng_state_cuda') is not None and DEVICE == 'cuda':
            torch.cuda.set_rng_state_all(bundle['rng_state_cuda'])

        # -- replay every captured microbatch exactly as the original
        #    GRAD_ACCUM loop did --
        for p in mdl.parameters():
            p.grad = None
        grad_accum = bundle.get('grad_accum', len(bundle['batches']))
        mdl.train()
        for xb, yb in bundle['batches']:
            x = torch.from_numpy(xb).to(DEVICE)
            y = torch.from_numpy(yb).to(DEVICE)
            if getattr(mdl, '_fock_capture', None) is not None:
                mdl._fock_capture = []  # drain the previous microbatch's entries
            loss, loss_ntp, v_reg, fock_reg = forward_with_vreg(
                x, y, LAMBDA_V, LAMBDA_FOCK_REG, FOCK_REG_EPS)
            if REGISTER_REPULSION:
                loss = loss + mdl.pop_repulsion_loss()
            (loss / grad_accum).backward()
            if getattr(mdl, '_fock_capture', None) is not None:
                # keyed by layer -- prefix_causal mode's _fock_layer_stats
                # already reports the last-position (full-prefix) slice, so
                # one dict per layer per microbatch is the expected shape.
                _fock_capture_per_microbatch.append(
                    {s['layer']: s for s in mdl._fock_capture})

        # -- (a) per-parameter grad norms (finer than per-group) --
        _param_norms = {n: float(p.grad.detach().norm())
                         for n, p in mdl.named_parameters() if p.grad is not None}
        _top_params = sorted(_param_norms.items(), key=lambda kv: kv[1],
                              reverse=True)[:top_k]
        _total_sq_excl, _total_sq_all = 0.0, 0.0
        for n, v in _param_norms.items():
            _total_sq_all += v * v
            _key, _ = assign_clip_group(n, _GRAD_CLIP_CFG)
            if _key not in WATCHDOG_EXCLUDE_GROUPS:
                _total_sq_excl += v * v
        _total_norm_excl = _total_sq_excl ** 0.5
        _total_norm_all = _total_sq_all ** 0.5
        _replayed_pg_norms = per_group_grad_norms(mdl, _GRAD_CLIP_CFG)

        _recorded = bundle['pre_clip_grad_norm']
        _fidelity_pct = 100 * abs(_total_norm_excl - _recorded) / max(_recorded, 1e-9)

        report = {
            'step': bundle['step'],
            'pre_clip_grad_norm_recorded': _recorded,
            'pre_clip_grad_norm_replayed_matching': _total_norm_excl,
            'pre_clip_grad_norm_replayed_all_groups': _total_norm_all,
            'fidelity_gap_pct': _fidelity_pct,
            'top_params': _top_params,
            'replayed_group_norms': _replayed_pg_norms,
            'layer_grad_norms': dict(sorted(_layer_grad_norms.items())),
            'activation_extremes': _act_extremes,
            'fock_capture_per_microbatch': _fock_capture_per_microbatch,
            'vtheta_exponent_hist': {'edges': list(_EXP_EDGES),
                                      'counts': {k: list(v)
                                                 for k, v in _exp_hist.items()}},
        }

        if verbose:
            print(f'\n[replay] fidelity check: replayed (matching-groups) total='
                  f'{_total_norm_excl:.1f}  vs. recorded pre_clip_grad_norm='
                  f'{_recorded:.1f}  (diff {_fidelity_pct:.1f}%)')
            if _fidelity_pct > 5.0:
                print('[replay][WARN] fidelity gap > 5% -- replay may not be '
                      'bit-exact (cuDNN nondeterminism, an unaccounted RNG '
                      'consumer, or a mismatched batch/weight pairing). '
                      'Treat the attribution below as approximate.')
            print(f'[replay] replayed total incl. reverse-channel groups='
                  f'{_total_norm_all:.1f}  (excl.={_total_norm_excl:.1f}; the gap '
                  f'is what the watchdog aggregate would have missed, SS33.3 Phase 0)')
            print('[replay] replayed per-group norms (top 8):')
            for k, v in sorted(_replayed_pg_norms.items(), key=lambda kv: kv[1],
                                reverse=True)[:8]:
                print(f'    {v:10.2f}  {k}')
            print('[replay] top per-parameter grad norms:')
            for n, v in _top_params:
                print(f'    {v:10.2f}  {n}')
            print('[replay] per-layer grad norm (gradient flowing INTO each layer boundary):')
            for ell, v in sorted(_layer_grad_norms.items()):
                print(f'    layer {ell:2d}: {v:10.2f}')
            print('[replay] forward-activation extremes at risky ops (by layer; '
                  '-1 = not layer-attributable):')
            for k, by_layer in _act_extremes.items():
                for ell, v in sorted(by_layer.items()):
                    print(f'    {k}  layer {ell:2d}: {v}')
            if _exp_hist:
                print('[replay] V_theta exponent occupancy per bank (token-slot '
                      'counts summed over all microbatches; "live" = exponent '
                      '> -10, where exp() >= 4.5e-5 and the well still '
                      'contributes gradient):')
                _lbl = [f'({_EXP_EDGES[b+1]:g},{_EXP_EDGES[b]:g}]'
                        for b in range(len(_EXP_EDGES) - 1)]
                print('    bank  ' + '  '.join(f'{s:>13}' for s in _lbl)
                      + f'  {"live_n":>10}  {"live_frac":>11}')
                for _bi in sorted(_exp_hist):
                    _c = _exp_hist[_bi]
                    _tot = sum(_c) or 1
                    _live = sum(_c[:_EXP_LIVE_BINS])
                    print(f'    {_bi:4d}  ' + '  '.join(f'{v:13d}' for v in _c)
                          + f'  {_live:10d}  {_live/_tot:11.4e}')
            if _fock_capture_per_microbatch:
                # 2026-08-30 (companion note SS38 follow-up): print the LAST
                # microbatch's per-layer register/gate health table -- the
                # one whose forward+backward is what pre_clip_grad_norm
                # above was actually measured on when grad_accum==1 (the
                # common case for these captures); earlier microbatches
                # (if grad_accum>1) are still in the returned report dict.
                _last_fc = _fock_capture_per_microbatch[-1]
                print(f'[replay] per-layer register/gate health '
                      f'(microbatch {len(_fock_capture_per_microbatch)-1} of '
                      f'{len(_fock_capture_per_microbatch)}, via set_fock_capture):')
                _fc_cols = ['active_frac', 'salience_mean', 'reg_cos_sim',
                            'destroy_mean', 'create_alpha_max', 'create_entropy',
                            'rev_entropy', 'rev_alpha_max', 'rev_scale',
                            'qforce_ratio']
                _fc_cols = [c for c in _fc_cols
                            if any(c in s for s in _last_fc.values())]
                print('    layer  ' + '  '.join(f'{c:>16}' for c in _fc_cols))
                for ell in sorted(_last_fc):
                    _row = _last_fc[ell]
                    print(f'    {ell:5d}  ' + '  '.join(
                        f'{_row.get(c, float("nan")):16.4g}' for c in _fc_cols))
    finally:
        # -- restore everything (non-pollution invariant), even on error --
        for _fn in reversed(_restore_fns):
            try:
                _fn()
            except Exception:
                pass
        mdl.load_state_dict(_saved_sd)
        _isolated_grad_restore(mdl, _saved_grads)
        torch.set_rng_state(_saved_rng_cpu)
        if _saved_rng_cuda is not None and DEVICE == 'cuda':
            torch.cuda.set_rng_state_all(_saved_rng_cuda)

    return report


def inspect_spike_tokens(step_tag, batch_idx=None, show_n=3, snippet_chars=240):
    """Decode the exact offending microbatch(es) captured at `step_tag` and
    print/return degeneracy stats (companion note SS38 follow-up SS38.4: no
    forward-activation signal separated the two localized events from the
    smooth-cascade ones, so this looks at the OTHER half of the forward
    pass -- what the model was actually reading -- for the handful of rows
    in the batch with the largest per-row max-token-repeat-run, on the
    theory that a degenerate run (e.g. a long whitespace/punctuation/
    boilerplate repeat straight out of OpenWebText) could be what triggers
    a localized early-layer blowup.

    This is pure CPU bookkeeping -- no model, no GPU, no RNG/weight state
    touched -- so it is safe to call at any time, interleaved with training
    or other replays, and does not require Cell 6 to have run at all
    (only CKPT_DIR/CKPT_PREFIX and the `tok` GPT-2 tokenizer from Cell 3).

    step_tag  : int or str, as in replay_spike_batch.
    batch_idx : which microbatch in the capture to inspect (default: all).
    show_n    : how many of the batch's rows (ranked by max-repeat-run,
                descending) to print a decoded snippet for.
    """
    path = CKPT_DIR / f'{CKPT_PREFIX}_step{step_tag}_spikebatch.pt'
    bundle = torch.load(path, map_location='cpu', weights_only=False)
    mb_indices = range(len(bundle['batches'])) if batch_idx is None else [batch_idx]
    all_rows = []
    for mb in mb_indices:
        xb, _yb = bundle['batches'][mb]
        for row in range(xb.shape[0]):
            ids = xb[row].tolist()
            n = len(ids)
            uniq = len(set(ids))
            # longest run of the same token id back-to-back.
            best_run, cur_run, cur_id = 1, 1, ids[0]
            for t in ids[1:]:
                if t == cur_id:
                    cur_run += 1
                else:
                    cur_run, cur_id = 1, t
                best_run = max(best_run, cur_run)
            all_rows.append({
                'microbatch': mb, 'row': row, 'seq_len': n,
                'unique_token_ratio': uniq / n, 'max_repeat_run': best_run,
                'ids': ids,
            })

    all_rows.sort(key=lambda r: r['max_repeat_run'], reverse=True)
    print(f'[tokens] step {bundle["step"]}: {len(all_rows)} row(s) across '
          f'{len(list(mb_indices))} microbatch(es), pre_clip_grad_norm='
          f'{bundle["pre_clip_grad_norm"]}')
    print(f'{"mb":>3} {"row":>4} {"len":>5} {"uniq_ratio":>10} {"max_repeat_run":>14}')
    for r in all_rows:
        print(f'{r["microbatch"]:3d} {r["row"]:4d} {r["seq_len"]:5d} '
              f'{r["unique_token_ratio"]:10.3f} {r["max_repeat_run"]:14d}')

    print(f'\n[tokens] decoded snippet for the top {min(show_n, len(all_rows))} '
          f'row(s) by max_repeat_run (most likely to show degenerate text):')
    for r in all_rows[:show_n]:
        text = tok.decode(r['ids'])
        snippet = text[:snippet_chars]
        print(f'\n  -- microbatch {r["microbatch"]}, row {r["row"]} '
              f'(len={r["seq_len"]}, unique_ratio={r["unique_token_ratio"]:.3f}, '
              f'max_repeat_run={r["max_repeat_run"]}) --')
        print('  ' + snippet.replace('\n', '\\n'))

    return all_rows


def attribute_spike_rows(step_tag, mdl=None, batch_idx=None,
                          track=('V_theta.depth_code',), verbose=True):
    """Per-row (per-sequence) gradient attribution for a captured spike.

    Companion note SS39. The premise: per-group clipping already caps
    `depth_code` at 0.25 on EVERY step, quiet or spiking (Cell 6's
    GRAD_CLIP_OVERRIDES, tightened 0.5 -> 0.25 on 2026-08-23 precisely
    because it was saturating its ceiling on every quiet step). So a
    localized event's applied `depth_code` update is exactly the same SIZE
    as a quiet step's -- 0.25 either way, whether the pre-clip norm was 88
    or 425. Its damage therefore cannot be magnitude; it has to be
    DIRECTION. The conjecture is that the direction gets dictated by a
    small minority of rows whose tokens land near a sharp V_theta well at
    layers 0-2 -- the only layers carrying meaningful salience
    (~0.32/0.14-0.22/0.06-0.15 at layers 0-2 vs. ~1e-3-1e-4 at layers 5-6,
    identically across all four replays so far, SS38.7).

    This tests that head-on: re-run the captured batch ONE ROW AT A TIME,
    each row scaled by 1/(grad_accum * rows_per_microbatch) so its number
    is its own share of the aggregate, and report per-row gradient norms
    for the tracked parameters plus the layer-0 h-boundary gradient. If the
    conjecture holds, the localized captures (39,983 / 41,837) should show
    one or two rows owning most of the total while the smooth-cascade
    captures (37,763 / 41,318) should look flat.

    Two honest caveats about what "per row" can mean here:

    1. The BAOAB O-step draws noise, so a row replayed alone does not see
       the noise realisation it saw inside its batch of 8. The RNG is
       therefore reset to the capture's pinned state before EVERY row, so
       all rows are compared under an identical draw -- which is what a
       relative attribution needs -- rather than each reproducing its exact
       in-batch contribution.
    2. Gradient norms don't add: ||sum_i g_i|| != sum_i ||g_i||. The
       concentration metric is therefore max_i ||g_i|| / sum_i ||g_i||,
       which sits at 1/n_rows when every row contributes equally and tends
       to 1 when a single row owns the gradient. ||sum_i g_i|| is also
       reported as a rough cross-check against replay_spike_batch's figure
       for the same parameter -- rough, not exact, because of caveat 1 and
       because the register-repulsion term (a parameter regulariser, not a
       per-row data term) is drained but excluded here.

    Same non-pollution invariant as replay_spike_batch: weights, .grad
    tensors and RNG state are snapshotted up front and restored in a
    finally block, so this is safe to interleave with training.
    """
    mdl = mdl if mdl is not None else model
    path = CKPT_DIR / f'{CKPT_PREFIX}_step{step_tag}_spikebatch.pt'
    bundle = torch.load(path, map_location='cpu', weights_only=False)
    print(f'[rows] loaded {path.name}  step={bundle["step"]}  '
          f'pre_clip_grad_norm={bundle["pre_clip_grad_norm"]}')

    _saved_grads = _isolated_grad_snapshot(mdl)
    _saved_sd = _copy.deepcopy(mdl.state_dict())
    _saved_rng_cpu = torch.get_rng_state()
    _saved_rng_cuda = torch.cuda.get_rng_state_all() if DEVICE == 'cuda' else None
    _restore_fns = []

    _layer0_grad = [float('nan')]
    _orig_layer_step = mdl._fock_layer_step

    def _instrumented_layer_step(h, h_prev, r, salience, m_b, gamma, dt,
                                  layer_idx, *args, **kwargs):
        out = _orig_layer_step(h, h_prev, r, salience, m_b, gamma, dt,
                                layer_idx, *args, **kwargs)
        h_new = out[0]
        if layer_idx == 0 and torch.is_tensor(h_new) and h_new.requires_grad:
            h_new.register_hook(
                lambda g: _layer0_grad.__setitem__(0, float(g.detach().norm())))
        return out

    mdl._fock_layer_step = _instrumented_layer_step
    _restore_fns.append(lambda: setattr(mdl, '_fock_layer_step', _orig_layer_step))

    _name2param = dict(mdl.named_parameters())
    _absent = [n for n in track if n not in _name2param]
    if _absent:
        print(f'[rows][WARN] not in named_parameters(), skipping: {_absent}')
    track = [n for n in track if n in _name2param]

    def _reset_rng():
        torch.set_rng_state(bundle['rng_state_cpu'])
        if bundle.get('rng_state_cuda') is not None and DEVICE == 'cuda':
            torch.cuda.set_rng_state_all(bundle['rng_state_cuda'])

    rows = []
    _accum = {n: None for n in track}
    try:
        mdl.load_state_dict(
            {k: v.to(DEVICE) for k, v in bundle['model_state_dict'].items()},
            strict=False)
        grad_accum = bundle.get('grad_accum', len(bundle['batches']))
        mb_indices = (list(range(len(bundle['batches']))) if batch_idx is None
                      else [batch_idx])
        mdl.train()
        for mb in mb_indices:
            xb, yb = bundle['batches'][mb]
            n_rows = xb.shape[0]
            for row in range(n_rows):
                for p in mdl.parameters():
                    p.grad = None
                _layer0_grad[0] = float('nan')
                _reset_rng()
                x = torch.from_numpy(xb[row:row + 1]).to(DEVICE)
                y = torch.from_numpy(yb[row:row + 1]).to(DEVICE)
                loss, loss_ntp, v_reg, fock_reg = forward_with_vreg(
                    x, y, LAMBDA_V, LAMBDA_FOCK_REG, FOCK_REG_EPS)
                if REGISTER_REPULSION:
                    mdl.pop_repulsion_loss()   # drained, deliberately not added
                (loss / (grad_accum * n_rows)).backward()
                _tot_sq = 0.0
                for p in mdl.parameters():
                    if p.grad is not None:
                        _tot_sq += float(p.grad.detach().norm()) ** 2
                _rec = {
                    'microbatch': mb, 'row': row,
                    'ntp': float(loss_ntp.detach()),
                    'layer0_h_grad': _layer0_grad[0],
                    'total_grad_norm': _tot_sq ** 0.5,
                }
                for n in track:
                    g = _name2param[n].grad
                    _rec[n] = float(g.detach().norm()) if g is not None else 0.0
                    if g is not None:
                        _accum[n] = (g.detach().clone() if _accum[n] is None
                                     else _accum[n] + g.detach())
                rows.append(_rec)
    finally:
        for _fn in reversed(_restore_fns):
            try:
                _fn()
            except Exception:
                pass
        mdl.load_state_dict(_saved_sd)
        _isolated_grad_restore(mdl, _saved_grads)
        torch.set_rng_state(_saved_rng_cpu)
        if _saved_rng_cuda is not None and DEVICE == 'cuda':
            torch.cuda.set_rng_state_all(_saved_rng_cuda)

    summary = {}
    for n in list(track) + ['layer0_h_grad', 'total_grad_norm']:
        vals = [r[n] for r in rows if r.get(n) is not None
                and not (isinstance(r[n], float) and math.isnan(r[n]))]
        _s = sum(vals)
        _ranked = sorted(vals, reverse=True)
        summary[n] = {
            'sum_of_row_norms': _s,
            'max_row_norm': _ranked[0] if _ranked else 0.0,
            'top1_share': (_ranked[0] / _s) if _s > 0 else float('nan'),
            'top3_share': (sum(_ranked[:3]) / _s) if _s > 0 else float('nan'),
            'uniform_share_baseline': (1.0 / len(vals)) if vals else float('nan'),
            'norm_of_summed_grad': (float(_accum[n].norm())
                                     if _accum.get(n) is not None else None),
        }

    report = {
        'step': bundle['step'],
        'pre_clip_grad_norm_recorded': bundle['pre_clip_grad_norm'],
        'rows': rows,
        'summary': summary,
    }

    if verbose:
        _key = track[0] if track else 'total_grad_norm'
        print(f'\n[rows] per-row attribution, ranked by {_key}  (each row scaled '
              f'by 1/(grad_accum*rows_per_mb), i.e. its own share of the aggregate):')
        _cols = ['ntp', 'layer0_h_grad', 'total_grad_norm'] + list(track)
        print(f'{"mb":>3} {"row":>4}  ' + '  '.join(f'{c:>20}' for c in _cols))
        for r in sorted(rows, key=lambda r: r.get(_key, 0.0), reverse=True):
            print(f'{r["microbatch"]:3d} {r["row"]:4d}  ' + '  '.join(
                f'{r.get(c, float("nan")):20.6g}' for c in _cols))
        for n, s in summary.items():
            print(f'\n[rows] concentration for {n}:')
            print(f'    top-1 row / sum-of-row-norms : {s["top1_share"]:.4f}'
                  f'   (flat-batch baseline {s["uniform_share_baseline"]:.4f})')
            print(f'    top-3 rows / sum-of-row-norms : {s["top3_share"]:.4f}')
            print(f'    sum of ||row grad||           : {s["sum_of_row_norms"]:.4f}')
            if s['norm_of_summed_grad'] is not None:
                print(f'    ||sum of row grads||          : '
                      f'{s["norm_of_summed_grad"]:.4f}   (rough cf. '
                      f'replay_spike_batch for this param; see caveats)')
    return report


def replay_precision_cap_ablation(step_tag, budgets=(1.0, 4.0, None), mdl=None,
                                   top_k=8, verbose=True):
    """Companion note SS41.6/SS41.7 item 1: re-run a captured spike batch
    under several `precision_lr_max` budgets -- weights, batch, and RNG
    state held bit-identical to the capture via the same snapshot/restore
    invariant `replay_spike_batch` uses -- to check whether capping
    sigma_max(B_k)^2 (model_aniso_gaussian_vtheta.py's `_bound_lowrank`)
    tames the exponent blow-up and the resulting grad-norm, BEFORE
    committing to resuming live training with the cap switched on.

    Mirrors `replay_integrator_ablation` (companion note SS40), but swaps
    `bank._precision_lr_max` across arms instead of `cfg.integrator` --
    `_bound_lowrank` reads `self._precision_lr_max` fresh on every forward
    call (it is a plain Python attribute, not a buffer or nn.Parameter),
    so this is a live-model attribute swap, not a new checkpoint.

    budgets : iterable of float or None. `None` means uncapped -- the
        value PRECISION_LR_MAX has actually been training with -- and
        should normally stay in the list as the baseline arm every other
        budget is compared against. A float `b` sets
        `sigma_max(B_k)^2 <= b` on EVERY V_theta bank
        (`bank._precision_lr_max = b`, via `mdl.V_theta.banks`) for the
        duration of that arm only.
    mdl : the live `model` by default.

    Same non-pollution invariant as replay_spike_batch: weights, .grad
    tensors, RNG state, and every bank's `_precision_lr_max` are
    snapshotted up front and restored in a `finally` block, so this is
    safe to call on the live model mid-training.

    Returns {arm_label: {budget, pre_clip_grad_norm, replayed_group_norms,
    per_layer_h_grad, vtheta_exponent_min}}.
    """
    mdl = mdl if mdl is not None else model
    path = CKPT_DIR / f'{CKPT_PREFIX}_step{step_tag}_spikebatch.pt'
    bundle = torch.load(path, map_location='cpu', weights_only=False)
    print(f'[precap] loaded {path.name}  step={bundle["step"]}  '
          f'pre_clip_grad_norm={bundle["pre_clip_grad_norm"]}')

    _saved_grads = _isolated_grad_snapshot(mdl)
    _saved_sd = _copy.deepcopy(mdl.state_dict())
    _saved_rng_cpu = torch.get_rng_state()
    _saved_rng_cuda = torch.cuda.get_rng_state_all() if DEVICE == 'cuda' else None
    _banks = mdl.V_theta.banks
    _saved_budgets = [b._precision_lr_max for b in _banks]
    _orig_layer_step = mdl._fock_layer_step
    _vt_hooks = []

    _layer_grad = {}

    def _instrumented_layer_step(h, h_prev, r, salience, m_b, gamma, dt,
                                  layer_idx, *args, **kwargs):
        out = _orig_layer_step(h, h_prev, r, salience, m_b, gamma, dt,
                                layer_idx, *args, **kwargs)
        h_new = out[0]
        if torch.is_tensor(h_new) and h_new.requires_grad:
            def _layer_hook(g, li=layer_idx):
                # NOTE: must not `return` the setdefault(...) result --
                # torch treats any non-None tensor-hook return value as a
                # gradient replacement, and dict.setdefault(...) returns
                # the (float) value, which crashes autograd with
                # "expected Variable, but hook returned 'float'".
                _layer_grad.setdefault(li, float(g.detach().norm()))
            h_new.register_hook(_layer_hook)
        return out

    # per-bank exponent min -- the direct check that the cap is actually
    # biting (bank._components() already applies _bound_lowrank internally,
    # so this reads the SAME B the forward pass used, capped or not).
    _exp_min = {}

    def _make_vtheta_hook(idx):
        def _vt_fwd_hook(module, args, kwargs, output):
            xi_in = args[0] if len(args) > 0 else kwargs.get('xis', kwargs.get('xi'))
            h_in = args[1] if len(args) > 1 else kwargs.get('h')
            try:
                mu, a, w, B = module._components(xi_in)
                h_e = h_in.unsqueeze(-2)
                diff = h_e - mu
                diag_term = (a * diff * diff).sum(dim=-1)
                if B.shape[-1] > 0:
                    Bt_diff = torch.einsum('...kd,...kdr->...kr', diff, B)
                    lr_term = (Bt_diff * Bt_diff).sum(dim=-1)
                else:
                    lr_term = torch.zeros_like(diag_term)
                exponent = -0.5 * (diag_term + lr_term)
                _cur = _exp_min.get(idx, float('inf'))
                _exp_min[idx] = min(_cur, float(exponent.detach().min()))
            except Exception:
                pass
        return _vt_fwd_hook

    results = {}
    try:
        mdl.load_state_dict(
            {k: v.to(DEVICE) for k, v in bundle['model_state_dict'].items()},
            strict=False)
        mdl._fock_layer_step = _instrumented_layer_step
        for _idx, _bank in enumerate(_banks):
            _h_vt = _bank.register_forward_hook(_make_vtheta_hook(_idx), with_kwargs=True)
            _vt_hooks.append(_h_vt)
        grad_accum = bundle.get('grad_accum', len(bundle['batches']))
        mdl.train()

        for budget in budgets:
            label = ('uncapped (None, as trained)' if budget is None
                      else f'precision_lr_max={budget}')
            for b in _banks:
                b._precision_lr_max = budget
            _layer_grad.clear()
            _exp_min.clear()
            for p in mdl.parameters():
                p.grad = None
            torch.set_rng_state(bundle['rng_state_cpu'])
            if bundle.get('rng_state_cuda') is not None and DEVICE == 'cuda':
                torch.cuda.set_rng_state_all(bundle['rng_state_cuda'])

            for xb, yb in bundle['batches']:
                x = torch.from_numpy(xb).to(DEVICE)
                y = torch.from_numpy(yb).to(DEVICE)
                loss, *_ = forward_with_vreg(
                    x, y, LAMBDA_V, LAMBDA_FOCK_REG, FOCK_REG_EPS)
                (loss / grad_accum).backward()

            _pg = per_group_grad_norms(mdl, _GRAD_CLIP_CFG)
            _total_sq_excl = sum(v * v for k, v in _pg.items()
                                   if k not in WATCHDOG_EXCLUDE_GROUPS)
            results[label] = {
                'budget': budget,
                'pre_clip_grad_norm': _total_sq_excl ** 0.5,
                'replayed_group_norms': _pg,
                'per_layer_h_grad': dict(sorted(_layer_grad.items())),
                'vtheta_exponent_min': dict(sorted(_exp_min.items())),
            }
            if verbose:
                _top = sorted(_pg.items(), key=lambda kv: kv[1], reverse=True)[:top_k]
                print(f'\n[precap] {label}:')
                print(f'    pre_clip_grad_norm={results[label]["pre_clip_grad_norm"]:.2f}')
                print('    top groups: '
                      + ', '.join(f'{k}={v:.2f}' for k, v in _top))
                print('    per-layer h-grad: '
                      + ', '.join(f'L{ell}={v:.4f}' for ell, v in
                                  results[label]['per_layer_h_grad'].items()))
                print('    V_theta exponent min per bank: '
                      + ', '.join(f'bank{idx}={v:.1f}' for idx, v in
                                  results[label]['vtheta_exponent_min'].items()))
    finally:
        mdl._fock_layer_step = _orig_layer_step
        for _h in _vt_hooks:
            _h.remove()
        for b, budget in zip(_banks, _saved_budgets):
            b._precision_lr_max = budget
        mdl.load_state_dict(_saved_sd)
        _isolated_grad_restore(mdl, _saved_grads)
        torch.set_rng_state(_saved_rng_cpu)
        if _saved_rng_cuda is not None and DEVICE == 'cuda':
            torch.cuda.set_rng_state_all(_saved_rng_cuda)

    if verbose and results:
        _recorded = bundle['pre_clip_grad_norm']
        _base_label = 'uncapped (None, as trained)'
        _base = results.get(_base_label, {}).get('pre_clip_grad_norm')
        print(f'\n[precap] summary (step={bundle["step"]}, recorded pre-clip='
              f'{_recorded:.1f} at capture time):')
        print(f'    {"arm":<32} {"pre_clip":>10}  {"vs uncapped":>12}')
        for label, r in results.items():
            _ratio = (f'{r["pre_clip_grad_norm"] / _base:.3f}x'
                       if _base else 'n/a')
            print(f'    {label:<32} {r["pre_clip_grad_norm"]:10.2f}  {_ratio:>12}')
        if _base_label not in results:
            print('    (no uncapped arm in `budgets` -- include None for a '
                  'like-for-like baseline)')
    return results


def replay_integrator_ablation(step_tag, lowrank_layers=frozenset({0, 1, 2}),
                                mdl=None, top_k=8, verbose=True):
    """Companion note SS40.4: re-run a captured spike batch under two
    integrator configs -- the recorded 'baoab_cfc' and 'baoab_cfc_lowrank'
    restricted to `lowrank_layers` -- with weights, batch, and RNG state
    held bit-identical to the capture, to check whether the exact
    (SVD-based) low-rank substep suppresses the localized-mode spike that
    plain `baoab_cfc`'s approximate off-diagonal kick produces, BEFORE
    committing to a live layer-restricted trial.

    Sibling of `replay_precision_cap_ablation`: same snapshot/restore
    invariant, but swaps `mdl.cfg.integrator` / `mdl.cfg.lowrank_layers`
    across arms instead of `bank._precision_lr_max` --
    `model_parf_multixi.py` reads both as plain runtime branches
    (`use_cfc = cfg.integrator == 'baoab_cfc'`, `use_lowrank = cfg.integrator
    == 'baoab_cfc_lowrank'`), so this is a live-model attribute swap, not a
    new checkpoint -- the weights (B_proj, mu_proj, etc.) are identical
    either way.

    lowrank_layers : which layers run the exact low-rank substep in the
        second arm. Defaults to {0, 1, 2} per SS38.7's salience profile
        (the localized mode lives almost entirely in L0-2; SS34.3's cost
        floor does not amortize cleanly with layer count, so wider sets
        cost more than proportionally).
    mdl : the live `model` by default.

    Same non-pollution invariant as replay_spike_batch: weights, .grad
    tensors, RNG state, and cfg.integrator / cfg.lowrank_layers are
    snapshotted up front and restored in a `finally` block, so this is
    safe to call on the live model mid-training.

    Returns {arm_label: {integrator, lowrank_layers, pre_clip_grad_norm,
    replayed_group_norms, per_layer_h_grad}}.
    """
    mdl = mdl if mdl is not None else model
    path = CKPT_DIR / f'{CKPT_PREFIX}_step{step_tag}_spikebatch.pt'
    bundle = torch.load(path, map_location='cpu', weights_only=False)
    print(f'[intgab] loaded {path.name}  step={bundle["step"]}  '
          f'pre_clip_grad_norm={bundle["pre_clip_grad_norm"]}')

    _saved_grads = _isolated_grad_snapshot(mdl)
    _saved_sd = _copy.deepcopy(mdl.state_dict())
    _saved_rng_cpu = torch.get_rng_state()
    _saved_rng_cuda = torch.cuda.get_rng_state_all() if DEVICE == 'cuda' else None
    _saved_integrator = mdl.cfg.integrator
    _saved_lowrank_layers = getattr(mdl.cfg, 'lowrank_layers', None)
    _orig_layer_step = mdl._fock_layer_step

    _layer_grad = {}

    def _instrumented_layer_step(h, h_prev, r, salience, m_b, gamma, dt,
                                  layer_idx, *args, **kwargs):
        out = _orig_layer_step(h, h_prev, r, salience, m_b, gamma, dt,
                                layer_idx, *args, **kwargs)
        h_new = out[0]
        if torch.is_tensor(h_new) and h_new.requires_grad:
            def _layer_hook(g, li=layer_idx):
                # NOTE: see replay_precision_cap_ablation's twin hook --
                # must not `return` the setdefault(...) result or torch's
                # autograd engine raises "expected Variable, but hook
                # returned 'float'".
                _layer_grad.setdefault(li, float(g.detach().norm()))
            h_new.register_hook(_layer_hook)
        return out

    _arms = [
        ('baoab_cfc as captured', 'baoab_cfc', None),
        (f'baoab_cfc_lowrank layers {sorted(lowrank_layers)}',
         'baoab_cfc_lowrank', lowrank_layers),
    ]
    results = {}
    try:
        mdl.load_state_dict(
            {k: v.to(DEVICE) for k, v in bundle['model_state_dict'].items()},
            strict=False)
        mdl._fock_layer_step = _instrumented_layer_step
        grad_accum = bundle.get('grad_accum', len(bundle['batches']))
        mdl.train()

        for label, integrator, layers in _arms:
            mdl.cfg.integrator = integrator
            mdl.cfg.lowrank_layers = layers
            _layer_grad.clear()
            for p in mdl.parameters():
                p.grad = None
            torch.set_rng_state(bundle['rng_state_cpu'])
            if bundle.get('rng_state_cuda') is not None and DEVICE == 'cuda':
                torch.cuda.set_rng_state_all(bundle['rng_state_cuda'])

            for xb, yb in bundle['batches']:
                x = torch.from_numpy(xb).to(DEVICE)
                y = torch.from_numpy(yb).to(DEVICE)
                loss, *_ = forward_with_vreg(
                    x, y, LAMBDA_V, LAMBDA_FOCK_REG, FOCK_REG_EPS)
                (loss / grad_accum).backward()

            _pg = per_group_grad_norms(mdl, _GRAD_CLIP_CFG)
            _total_sq_excl = sum(v * v for k, v in _pg.items()
                                   if k not in WATCHDOG_EXCLUDE_GROUPS)
            results[label] = {
                'integrator': integrator,
                'lowrank_layers': layers,
                'pre_clip_grad_norm': _total_sq_excl ** 0.5,
                'replayed_group_norms': _pg,
                'per_layer_h_grad': dict(sorted(_layer_grad.items())),
            }
            if verbose:
                _top = sorted(_pg.items(), key=lambda kv: kv[1], reverse=True)[:top_k]
                print(f'\n[intgab] {label}:')
                print(f'    pre_clip_grad_norm={results[label]["pre_clip_grad_norm"]:.2f}')
                print('    top groups: '
                      + ', '.join(f'{k}={v:.2f}' for k, v in _top))
                print('    per-layer h-grad: '
                      + ', '.join(f'L{ell}={v:.4f}' for ell, v in
                                  results[label]['per_layer_h_grad'].items()))
    finally:
        mdl._fock_layer_step = _orig_layer_step
        mdl.cfg.integrator = _saved_integrator
        mdl.cfg.lowrank_layers = _saved_lowrank_layers
        mdl.load_state_dict(_saved_sd)
        _isolated_grad_restore(mdl, _saved_grads)
        torch.set_rng_state(_saved_rng_cpu)
        if _saved_rng_cuda is not None and DEVICE == 'cuda':
            torch.cuda.set_rng_state_all(_saved_rng_cuda)

    if verbose and len(results) == 2:
        _recorded = bundle['pre_clip_grad_norm']
        _labels = list(results.keys())
        _base = results[_labels[0]]['pre_clip_grad_norm']
        print(f'\n[intgab] summary (step={bundle["step"]}, recorded pre-clip='
              f'{_recorded:.1f} at capture time):')
        print(f'    {"arm":<42} {"pre_clip":>10}  {"vs baoab_cfc":>12}')
        for label, r in results.items():
            _ratio = f'{r["pre_clip_grad_norm"] / _base:.3f}x' if _base else 'n/a'
            print(f'    {label:<42} {r["pre_clip_grad_norm"]:10.2f}  {_ratio:>12}')
    return results


def replay_all_captures(mdl=None, top_k=5, verbose=False):
    """Replay every '_spikebatch.pt' currently on disk and print one
    summary attribution table across all of them.

    Companion note SS36: with CAPTURE_SPIKE_THRESHOLD (Cell 6) decoupled
    from the rare GRAD_NORM_HARD_TRIGGER, Phase 1 now harvests the far
    more frequent moderate-band spikes (the ones actually correlated with
    the run's convergence plateau, not just the occasional catastrophic
    reload). A consistent leading-group/per-layer signature across many
    of these is much stronger evidence than any single replay -- this is
    the batch version of calling replay_spike_batch(step_tag) by hand for
    every capture.

    Set verbose=True to also print each individual replay's full report
    (per-parameter / per-layer / activation-extreme detail); default only
    prints the summary table below.
    """
    mdl = mdl if mdl is not None else model
    _prefix = f'{CKPT_PREFIX}_step'
    _suffix = '_spikebatch.pt'
    paths = sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_spikebatch.pt'))
    if not paths:
        print('[replay-all] no captures found on disk.')
        return []
    reports = []
    for p in paths:
        step_tag = int(p.name[len(_prefix):-len(_suffix)])
        print(f'[replay-all] replaying step {step_tag} ({p.name}) ...')
        try:
            reports.append(replay_spike_batch(step_tag, mdl=mdl, top_k=top_k,
                                                verbose=verbose))
        except KeyboardInterrupt:
            # 2026-08-30 (companion note SS37): KeyboardInterrupt is a
            # BaseException, not an Exception, so the `except Exception`
            # below never caught it -- it used to propagate all the way to
            # IPython's top-level uncaught-exception handler, which pins
            # the interrupted replay's entire forward/backward graph via
            # sys.last_traceback (the same OOM failure mode as interrupting
            # Cell 6 itself, just triggered from in here instead). Catching
            # it here keeps that graph scoped to this frame, freed on
            # return, same fix as run_training() in Cell 6.
            print(f'\n[replay-all] interrupted while replaying step {step_tag}; '
                  f'stopping here (kept {len(reports)} report(s) already '
                  f'replayed). Run the GPU memory-cleanup cell before '
                  f'retrying, just in case.')
            break
        except Exception as e:
            print(f'[replay-all][WARN] failed on {p.name}: {e}')

    print(f'\n[replay-all] summary across {len(reports)} captured spikes '
          f'(sorted by step):')
    print(f'{"step":>8} {"pre_clip":>9} {"fidelity%":>10}  leader (norm)'
          f'  |  next-2 groups')
    for r in sorted(reports, key=lambda r: r['step']):
        _pg = sorted(r['replayed_group_norms'].items(), key=lambda kv: kv[1],
                      reverse=True)
        _leader = f'{_pg[0][0]}={_pg[0][1]:.1f}' if _pg else '(none)'
        _next2 = '  '.join(f'{k}={v:.1f}' for k, v in _pg[1:3])
        print(f'{r["step"]:8d} {r["pre_clip_grad_norm_recorded"]:9.1f} '
              f'{r["fidelity_gap_pct"]:10.4f}  {_leader:<28}  |  {_next2}')
    return reports


print('Cell 6d ready (helpers defined; call sites need Cell 6 to have at least')
print('started -- see the usage note atop this cell): e.g. replay_spike_batch(34091)')
print('              or replay_all_captures() to replay + summarize every capture on disk')
print('              or attribute_spike_rows(34091) for per-row depth_code attribution')
print('              or inspect_spike_tokens(34091) to decode the offending microbatch')
print('              (inspect_spike_tokens needs only Cell 3\'s `tok`, not Cell 6 at all)')
print('              or replay_precision_cap_ablation(34091, budgets=[1.0, 4.0, None])')
print('              to test precision_lr_max caps before enabling it live (SS41.6/41.7)')
print('              or replay_integrator_ablation(34091, lowrank_layers={0,1,2})')
print('              to test baoab_cfc vs baoab_cfc_lowrank on the same capture (SS40.4)')
print('Available captures on disk:')
for _p in sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_spikebatch.pt')):
    print(f'  {_p.name}')


In [ ]:
# == Cell 6: Training loop =============================================

LR            = 3e-4
WEIGHT_DECAY  = 0.01
WARMUP_STEPS  = int(WSD_WARMUP_FRAC * TOTAL_STEPS) if LR_SCHEDULE == 'wsd' else 4000
GRAD_CLIP     = 1.0
GRAD_CLIP_VPHI = 0.3

PER_GROUP_CLIP = True
GRAD_CLIP_OVERRIDES = {
    'V_phi': GRAD_CLIP_VPHI,
    'creation_gate': 0.3,
    'destruction_gate': 0.3,
    'reverse_channel_scale': 0.1,
    'reverse_ch': 0.1,
    'register': 0.3,
    # 2026-08-23: tightened 0.5 -> 0.25. depth_code was already saturating
    # its old ceiling on *every* quiet step (top[override:depth_code] ~
    # 0.5-1.6) and was the single largest pre-clip contributor (1592.8 at
    # the worst step-6435 spike) in the first CfC/BAOAB grad-clip burst
    # seen on the g0.1 OWT run (steps 6297-6616, val_ppl 176.88->207.11).
    # Halving its ceiling costs little in the already-saturated quiet
    # regime and caps its burst contribution proportionally.
    'depth_code': 0.25,
}
# NOTE: excluding reverse_channel_scale/reverse_ch keeps the watchdog EMA
# from false-triggering on their own normal warmup ramp, but it also means
# the EMA below is structurally blind to them -- they showed up in the
# top-4 of nearly every spike in the 2026-08-23 burst (e.g.
# reverse_channel_scale=403.8, reverse_ch=363.1 at step 6435). Left as-is
# for now; GRAD_NORM_HARD_TRIGGER below is a fallback that does see them.
WATCHDOG_EXCLUDE_GROUPS = {'override:reverse_channel_scale', 'override:reverse_ch'}

GRAD_SPIKE_DEBUG     = True
GRAD_SPIKE_THRESHOLD = 100.0
GRAD_SPIKE_COOLDOWN  = 0
EVAL_INTERVAL = 500
EVAL_ITERS    = 40
LOG_INTERVAL  = 50
CAUSAL_PROBE_INTERVAL = 10_000
TRAINED_LEAK_PROBE_INTERVAL = 10_000
TRAINED_LEAK_PROBE_K = 256
TRAINED_LEAK_PROBE_PAIRS = 2

if WSD_LR_FLOOR is None:
    WSD_LR_FLOOR = LR * 0.05

GRAD_NORM_EMA_ALPHA = 0.05
GRAD_NORM_EMA_THRESHOLD = 50.0
GRAD_NORM_EMA_PATIENCE = 200
# Fast, EMA-independent safety net added 2026-08-23. With alpha=0.05 and
# patience=200, an isolated spike decays back out of the EMA within ~15-20
# steps and never accumulates 200 *consecutive* above-threshold steps, so
# a short burst of huge-but-brief spikes (pre-clip total=2401.6 at step
# 6435, =1815.9 at step 6407, ...) can do lasting damage (val_ppl
# 176.88->207.11 over the 6000->6500 eval window) without ever tripping
# the patience-gated reload. This reloads best immediately on any single
# step whose raw (pre-EMA) grad_norm exceeds it, regardless of EMA/patience
# state. Set to None to disable.
# 2026-08-30 Stage-1 smoke test value, used once to validate Phase 1/2
# end-to-end (see companion note SS35) -- restored to the production
# value below afterward. Left here for reference, not active.
# GRAD_NORM_HARD_TRIGGER = 110.0
GRAD_NORM_HARD_TRIGGER = 500.0


# How many '_prereload' snapshots (the about-to-be-discarded state AT the
# moment the watchdog fires, before it is overwritten by the last best)
# to keep on disk at once; 0 disables the feature entirely. A dense
# reload cluster can produce a lot of these in a short span, so this
# rotates rather than growing unbounded -- see docs/Example_Stiffness_
# Audit_OWT_g0.1_Anisotropic_Gaussian.md sec 9 in the SCAF repo for why
# this exists: without it, a run whose best_ppl stops improving for
# thousands of steps during a bad cluster leaves NO checkpoint at all in
# that window for a later stiffness audit to look at, because both of
# this file's other save triggers (a new best; the CKPT_INTERVAL grid)
# can go arbitrarily long without firing precisely while the cluster is
# active. This one is unconditional on val_ppl, so it cannot have that
# failure mode.
PRERELOAD_SNAPSHOT_MAX_KEEP = 5

# Root-cause diagnostic Phase 1 (CfC/BAOAB companion note SS33.3): alongside
# the '_prereload' snapshot, also capture the PRE-optim.step() weights, the
# exact microbatch(es), and the torch RNG state for any step whose pre-clip
# grad norm crosses CAPTURE_SPIKE_THRESHOLD, so the offending forward+
# backward can be replayed deterministically in isolation (Phase 2).
#
# 2026-08-30 (companion note SS36): CAPTURE_SPIKE_THRESHOLD is deliberately
# a SEPARATE, lower knob from GRAD_NORM_HARD_TRIGGER below, not the same
# value -- the two answer different questions. GRAD_NORM_HARD_TRIGGER is
# 'how bad does a step have to be before we discard progress and reload
# the best checkpoint' (rare, >=500 in this run's history); the plateau
# analysis in SS36 showed the run can sit stuck for 1,000+ steps with
# ZERO hard triggers, driven instead by the far more frequent 100-450
# 'grad_spike' band (GRAD_SPIKE_THRESHOLD above), which is the actual
# convergence question. Gating capture on GRAD_NORM_HARD_TRIGGER meant
# Phase 1 could only ever harvest the rare, possibly-unrepresentative
# catastrophic events; gating it on this separate, lower threshold
# instead harvests the frequent moderate band WITHOUT touching the
# reload behaviour at all (CAPTURE_SPIKE_THRESHOLD plays no role in the
# watchdog logic below, so lowering it cannot cause extra reloads).
CAPTURE_SPIKE_THRESHOLD = 200.0
# Only the hard trigger and this threshold have a single well-defined
# 'offending batch' to snapshot; the slow EMA trigger is a drift, not a
# one-step event, and stays a Phase-0 log-mining question. Same rotation
# policy as PRERELOAD_SNAPSHOT_MAX_KEEP; 0/False disables.
CAPTURE_SPIKE_BATCH = True
# Raised 5 -> 12 (SS36): with capture now firing on the frequent moderate
# band rather than only the rare hard trigger, a deeper rotation keeps a
# small *library* of captures on disk to replay across (see the new
# replay_all_captures() in Cell 6d) instead of just the single latest one.
SPIKEBATCH_SNAPSHOT_MAX_KEEP = 12

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)


def lr_schedule(step):
    if LR_SCHEDULE == 'wsd':
        warmup_end = int(WSD_WARMUP_FRAC * TOTAL_STEPS)
        stable_end = int((WSD_WARMUP_FRAC + WSD_STABLE_FRAC) * TOTAL_STEPS)
        if step < warmup_end:
            return LR * (step + 1) / max(warmup_end, 1)
        elif step < stable_end:
            return LR
        else:
            decay_steps = TOTAL_STEPS - stable_end
            progress = (step - stable_end) / max(decay_steps, 1)
            cos_decay = 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))
            return WSD_LR_FLOOR + (LR - WSD_LR_FLOOR) * cos_decay
    else:
        if step < WARMUP_STEPS:
            return LR * (step + 1) / WARMUP_STEPS
        progress = (step - WARMUP_STEPS) / max(TOTAL_STEPS - WARMUP_STEPS, 1)
        return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def fock_coupling_reg(mdl, lam, eps):
    alphas = mdl.xi_module.alpha
    return -lam * torch.log(alphas + eps).sum()


def forward_with_vreg(x, targets, lambda_v, lambda_fock, fock_eps):
    h0 = model._embed(x)
    h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
    logits = model.compute_logits(h_L)
    loss_ntp = F.cross_entropy(
        logits.reshape(-1, model_cfg.vocab_size),
        targets.reshape(-1),
    )
    v_reg_value = torch.tensor(0.0, device=x.device)
    fock_reg_value = torch.tensor(0.0, device=x.device)
    loss = loss_ntp

    if lambda_v > 0:
        xis = model.xi_module(h_L.detach())
        V_vals = model.V_theta(xis, h_L)
        v_reg_value = (V_vals.float() ** 2).mean()
        loss = loss + lambda_v * v_reg_value

    if lambda_fock > 0:
        fock_reg_value = fock_coupling_reg(model, lambda_fock, fock_eps)
        loss = loss + fock_reg_value

    return loss, loss_ntp, v_reg_value, fock_reg_value


@torch.no_grad()
def evaluate():
    model.eval()
    # 2026-08-30 (companion note SS37): try/finally so an interrupt landing
    # mid-eval (e.g. while inspecting spike captures) can't leave `model`
    # stuck in .eval() mode for the run_training() call that resumes after it.
    try:
        losses = []
        _mem_pre_gb = torch.cuda.memory_allocated() / 1e9
        torch.cuda.reset_peak_memory_stats()
        _mem_per_iter = []
        for _i in range(EVAL_ITERS):
            xb, yb = get_batch(val_ids, BATCH_SIZE, BLOCK_SIZE, rng)
            x = torch.from_numpy(xb).to(DEVICE)
            y = torch.from_numpy(yb).to(DEVICE)
            with torch.enable_grad():
                _, loss = model(x, y)
            losses.append(loss.item())
            del loss, x, y
            if _i % 10 == 0 or _i == EVAL_ITERS - 1:
                _mem_per_iter.append(torch.cuda.memory_allocated() / 1e9)
        _mem_post_gb = torch.cuda.memory_allocated() / 1e9
        _mem_peak_gb = torch.cuda.max_memory_allocated() / 1e9
        _mem_trend = ', '.join(f'{m:.2f}' for m in _mem_per_iter)
        print(f'    [evaluate] mem before={_mem_pre_gb:.2f}GB  '
              f'after={_mem_post_gb:.2f}GB  peak_during={_mem_peak_gb:.2f}GB  '
              f'trend(every 10 iters)=[{_mem_trend}]GB')
    finally:
        model.train()
    return float(np.mean(losses))


def run_causal_probe(step_num):
    import math as _math
    # Use the same V_theta family and the same integrator as the run, so
    # this probe certifies prefix-causality for what is actually training.
    from model_aniso_gaussian_vtheta import (
        AnisotropicDepthConditionedGaussianVTheta as _DCMCGVT,
        install_aniso_depth_routing as _idr,
    )
    _PROBE_VOCAB, _PROBE_D, _PROBE_L = 101, 32, 4
    _PROBE_T, _PROBE_M, _PROBE_XI = 48, 8, 3
    _PROBE_WELLS = 4

    _logfreq_probe = Path('/tmp/causal_probe_logfreq.npy')
    np.save(_logfreq_probe, np.full(_PROBE_VOCAB, 5.0, dtype=np.float32))

    _probe_cfg = FockMultiXiPARFConfig(
        vocab_size=_PROBE_VOCAB, d=_PROBE_D, max_len=64, L=_PROBE_L,
        v_hidden=64, v_depth=3, dt=1.0,
        mass_mode='logfreq', logfreq_path=str(_logfreq_probe),
        logfreq_init_alpha=0.1, init_gamma=1.0, fixed_gamma=0.30,
        causal_force=True, ln_after_step=True,
        xi_channels=_PROBE_XI, xi_alpha_inits=[0.5, 0.9, 0.99],
        xi_learnable=True, xi_alpha_init_mode='explicit',
        v_phi_kind='structural_competitive',
        v_phi_d_type=8, v_phi_d_angle=4, v_phi_eps=0.1,
        v_phi_phi_hidden=16, v_phi_theta_hidden=16, v_phi_mlp_hidden=16,
        top_k=8, v_phi_n_heads=2,
        use_output_bias=True, tie_embeddings=False,
        score_head_hidden=8,
        gumbel_tau_init=1.0, gumbel_tau_min=0.3, gumbel_noise=True,
        use_gathered_v_phi=True, use_layer_checkpoint=False,
        ln_before_distance=True, per_layer_v_phi_scale=True,
        fock_version='v2', n_registers=_PROBE_M,
        register_salience_decay=0.5, register_salience_threshold=0.005,
        creation_gate_hidden=16, stack_discipline=True,
        d_k=16, tau_create_init=8.0,
        reverse_channel=True, reverse_channel_stable=True,
        reverse_channel_pre_ln=True, reverse_channel_soft_norm=True,
        reverse_channel_warmup_steps=4000, reverse_channel_per_layer=True,
        per_register_tau=True, per_register_keys=True,
        ortho_register_init=True, register_repulsion=False,
        prefix_causal_registers=True,
        integrator=CFG_INTEGRATOR,
        vtheta_analytic_force=CFG_VTHETA_ANALYTIC,
        lowrank_max_modes=LOWRANK_MAX_MODES,
        langevin_T=0.0,          # noise would swamp the leak signal
    )
    torch.manual_seed(1234)
    _probe_model = FockMultiXiPARFLM(_probe_cfg)
    _probe_model.V_theta = _DCMCGVT(
        d=_PROBE_D, K=_PROBE_WELLS, n_ctx=_PROBE_XI, n_layers=_PROBE_L,
        rank=2, w_scale=1.0, init_log_precision=-_math.log(_PROBE_D),
        precision_max=2.0/_PROBE_D, precision_lr_max=PRECISION_LR_MAX,
        code_init_std=0.02,
    )
    _idr(_probe_model)
    _probe_model.double().eval()

    with torch.no_grad():
        _probe_model.reverse_channel_scale.fill_(1.0)
        _probe_model.reverse_warmup_step.fill_(4000)

    _t_p = _PROBE_T // 2
    _prng = np.random.default_rng(7)
    _x1 = torch.from_numpy(_prng.integers(0, _PROBE_VOCAB, (2, _PROBE_T))).long()
    _x2 = _x1.clone()
    _x2[:, _t_p:] = torch.from_numpy(
        _prng.integers(0, _PROBE_VOCAB, (2, _PROBE_T - _t_p))).long()

    with torch.enable_grad():
        _la = _probe_model(_x1)[0].detach()
        _lb = _probe_model(_x2)[0].detach()
    _max_delta = float((_la[:, :_t_p] - _lb[:, :_t_p]).abs().max().item())

    _probe_model.train()
    torch.manual_seed(99)
    with torch.enable_grad():
        _lta = _probe_model(_x1)[0].detach()
    torch.manual_seed(99)
    with torch.enable_grad():
        _ltb = _probe_model(_x2)[0].detach()
    _max_delta = max(_max_delta,
                     float((_lta[:, :_t_p] - _ltb[:, :_t_p]).abs().max().item()))

    _passed = (_max_delta == 0.0)
    del _probe_model, _la, _lb, _lta, _ltb, _x1, _x2
    gc.collect()

    status = 'PASS' if _passed else '*** FAIL ***'
    print(f'\n[causal probe] step {step_num:,}  max|dlogit|={_max_delta:.3e}  [{status}]')
    if not _passed:
        print('[causal probe] WARNING: nonzero future sensitivity detected!')
    return _passed, _max_delta


def run_trained_leak_probe(step_num):
    _debug_dir = str(CA_DIR / 'scaleup' / 'debug')
    if _debug_dir not in sys.path:
        sys.path.insert(0, _debug_dir)
    from fock_trained_leak_probe import probe_trained_leak, honest_ppl_test

    print(f'\n{"="*64}')
    print(f'[trained leak probe] step {step_num:,} -- running on live model')
    print(f'{"="*64}')

    probe_res = probe_trained_leak(
        model, val_ids, device=DEVICE, context=BLOCK_SIZE,
        n_pairs=TRAINED_LEAK_PROBE_PAIRS, use_float64=False)
    honest_res = honest_ppl_test(
        model, val_ids, k=TRAINED_LEAK_PROBE_K,
        context=BLOCK_SIZE, batch=BATCH_SIZE, device=DEVICE)
    model.train()

    result = {
        'step': step_num,
        'probe_max_dlogit_past': probe_res['max_dlogit_past'],
        'probe_mean_dnll_past_nats': round(probe_res['mean_dnll_past'], 6),
        'probe_gate_zero_control': probe_res['gate_zero_control'],
        'honest_k': honest_res['k'],
        'ppl_mid_window_standard': round(honest_res['ppl_mid_window'], 4),
        'ppl_last_pos_leak_free': round(honest_res['ppl_last_pos'], 4),
        'paired_diff_nats': round(honest_res['paired_diff_nats'], 6),
        'paired_diff_se': round(honest_res['paired_diff_se'], 6),
    }

    _leak_status = 'CLEAN' if result['paired_diff_nats'] < 0.1 else 'LEAK DETECTED'
    print(f'\n[trained leak probe] step {step_num:,}  '
          f'honest_PPL={result["ppl_last_pos_leak_free"]:.2f}  '
          f'standard_PPL={result["ppl_mid_window_standard"]:.2f}  '
          f'diff={result["paired_diff_nats"]:+.4f} nats  [{_leak_status}]')
    return result


def save_checkpoint(step_num, val_loss_val, tag_suffix=''):
    ckpt = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optim.state_dict(),
        'model_cfg': asdict(model_cfg),
        'train_cfg': {
            'batch_size': BATCH_SIZE, 'block_size': BLOCK_SIZE,
            'grad_accum': GRAD_ACCUM, 'effective_batch': EFFECTIVE_BATCH,
            'steps': TOTAL_STEPS, 'lr': LR, 'weight_decay': WEIGHT_DECAY,
            'warmup_steps': WARMUP_STEPS, 'grad_clip': GRAD_CLIP,
            'grad_clip_vphi': GRAD_CLIP_VPHI,
            'optimizer': OPTIMIZER, 'grad_centralization': GRAD_CENTRALIZATION,
            'lambda_v': LAMBDA_V, 'v_theta_variant': V_THETA_VARIANT,
            'lr_schedule': LR_SCHEDULE,
            'v_theta_n_heads': V_THETA_N_HEADS,
            'v_theta_wells_per_head': V_THETA_WELLS_PER_HEAD,
            'v_theta_depth_condition': V_THETA_DEPTH_CONDITION,
            'v_theta_depth_code_init_std': V_THETA_DEPTH_CODE_INIT_STD,
            'aniso_rank': ANISO_RANK,
            'lambda_fock_reg': LAMBDA_FOCK_REG,
            'integrator': INTEGRATOR,
            'cfg_integrator': CFG_INTEGRATOR,
            'vtheta_analytic_force': CFG_VTHETA_ANALYTIC,
            'langevin_T': LANGEVIN_T,
        },
        'step': step_num,
        'val_loss': val_loss_val,
        'val_ppl': math.exp(val_loss_val),
        'gamma': model.gamma.item(),
        'xi_alphas': model.xi_alpha_values(),
        'variant': (f'fock_parf_multixi_v2.1_aniso_gaussian_'
                f'dcvt{V_THETA_N_HEADS}_{INTEGRATOR}'),
        'corpus': 'openwebtext',
        'phase': 7,
        'seed': SEED,
    }
    fname = f'{CKPT_PREFIX}_step{step_num}{tag_suffix}.pt'
    path = CKPT_DIR / fname
    for _attempt in range(2):
        try:
            torch.save(ckpt, path)
            break
        except OSError as _e:
            if _e.errno == 107 and _attempt == 0:
                print(f'[WARN] Drive transport error saving checkpoint; remounting... ({_e})')
                try:
                    from google.colab import drive as _drv
                    _drv.mount('/content/drive', force_remount=True)
                except Exception as _re:
                    print(f'[WARN] Drive remount failed: {_re}')
                    print(f'[WARN] Checkpoint NOT saved: {path}')
                    return None
            else:
                print(f'[WARN] Checkpoint save failed: {_e}')
                return None
    print(f'  Checkpoint saved: {path}  (PPL={math.exp(val_loss_val):.2f})')
    if '_best' in tag_suffix:
        canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
        import shutil
        shutil.copy2(path, canonical)
        print(f'  Canonical best: {canonical}')
    return path


# -- Optimizer --
_trainable = [p for p in model.parameters() if p.requires_grad]
if OPTIMIZER == 'adamw':
    optim = torch.optim.AdamW(_trainable, lr=LR,
                              weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
elif OPTIMIZER == 'lamb':
    try:
        import torch_optimizer
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch_optimizer'])
        import torch_optimizer
    optim = torch_optimizer.Lamb(_trainable, lr=LR,
                                weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
elif OPTIMIZER == 'lion':
    try:
        from lion_pytorch import Lion
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lion-pytorch'])
        from lion_pytorch import Lion
    optim = Lion(_trainable, lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.99))
else:
    raise ValueError(f'Unknown OPTIMIZER={OPTIMIZER!r}; choose adamw / lamb / lion')
print(f'Optimizer: {type(optim).__name__}')
if DEVICE == 'cuda':
    _free, _total = torch.cuda.mem_get_info()
    print(f'  CUDA after optimiser: {_free/1e9:.1f} GB free / {_total/1e9:.1f} GB')

# -- Resume --
if resume_ckpt is not None and resume_step < TOTAL_STEPS:
    print(f'Resuming from checkpoint at step {resume_step:,}: {resume_ckpt}')
    ckpt_data = torch.load(resume_ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt_data['model_state_dict'], strict=False)
    if (REVERSE_CHANNEL and REVERSE_CHANNEL_STABLE and REVERSE_CHANNEL_RESET_SCALE
            and getattr(model, 'reverse_channel_scale', None) is not None):
        with torch.no_grad():
            model.reverse_channel_scale.zero_()
            if hasattr(model, 'reverse_warmup_step'):
                model.reverse_warmup_step.zero_()
        print('  [E5c] reverse_channel_scale re-zeroed + warmup reset')
    if 'optimizer_state_dict' in ckpt_data:
        try:
            optim.load_state_dict(ckpt_data['optimizer_state_dict'])
            print('  Optimizer state restored.')
        except (ValueError, KeyError) as e:
            print(f'  [info] Optimizer state incompatible, starting fresh: {e}')
    prev_ppl = ckpt_data.get('val_ppl', float('nan'))
    print(f'  Model loaded. Previous PPL: {prev_ppl:.2f}')
    del ckpt_data
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

# -- Training state --
log_path = RESULTS_DIR / 'training_log.jsonl'
_log_fh = [log_path.open('a')]
_log_write_count = [0]
# Google Drive's FUSE mount buffers writes locally and only reliably syncs
# them to Drive on file-descriptor close. A long training run keeps a single
# handle open for its whole (up to 24h) session, so if Colab kills the
# runtime abruptly (session timeout, disconnect, OOM) any writes since the
# last close can be silently lost even though flush() succeeded locally.
# Forcing an fsync + periodic close/reopen bounds how much log history can
# be lost to roughly _LOG_REOPEN_EVERY * LOG_INTERVAL steps.
_LOG_REOPEN_EVERY = 10


def _log_write(record_str):
    for _attempt in range(2):
        try:
            _log_fh[0].write(record_str)
            _log_fh[0].flush()
            try:
                os.fsync(_log_fh[0].fileno())
            except OSError:
                pass  # fsync isn't guaranteed to be meaningful on FUSE mounts
            _log_write_count[0] += 1
            if _log_write_count[0] % _LOG_REOPEN_EVERY == 0:
                _log_fh[0].close()
                _log_fh[0] = log_path.open('a')
            return
        except OSError as _e:
            if _e.errno == 107 and _attempt == 0:
                print(f'[WARN] Drive transport error on log write; remounting...')
                try:
                    from google.colab import drive as _drv
                    _drv.mount('/content/drive', force_remount=True)
                    try:
                        _log_fh[0].close()
                    except Exception:
                        pass
                    _log_fh[0] = log_path.open('a')
                except Exception as _re:
                    print(f'[WARN] Drive remount failed: {_re}; log record lost.')
                    return
            else:
                print(f'[WARN] Log write failed (attempt {_attempt+1}): {_e}')
                return


t0 = time.time()
model.train()
run_ntp = 0.0
run_vreg = 0.0
run_fock_reg = 0.0
n_run = 0
n_skipped = 0

best_val_ppl = float('inf')
_best_ckpt_path = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'

if not _best_ckpt_path.exists():
    _step_bests = sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_best.pt'))
    if _step_bests:
        _best_ckpt_path = _step_bests[-1]
        import shutil
        _canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
        shutil.copy2(_best_ckpt_path, _canonical)
        _best_ckpt_path = _canonical

if _best_ckpt_path.exists():
    try:
        _bd = torch.load(_best_ckpt_path, map_location='cpu', weights_only=False)
        best_val_ppl = _bd.get('val_ppl', float('inf'))
        print(f'Restored running best PPL: {best_val_ppl:.2f}')
        del _bd
    except Exception as e:
        print(f'[warn] {e}')

_grad_norm_ema = 0.0
_grad_norm_above_thresh = 0


def _reload_best(pre_reload_step=None):
    if pre_reload_step is not None and PRERELOAD_SNAPSHOT_MAX_KEEP > 0:
        # Snapshot the state that TRIGGERED this reload before it gets
        # overwritten -- this is the state a post-hoc stiffness audit
        # actually wants (see the config comment above for why the other
        # two save triggers cannot be relied on to have captured it).
        save_checkpoint(pre_reload_step, float('nan'), tag_suffix='_prereload')
        _existing = sorted(
            CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_prereload.pt'),
            key=lambda p: p.stat().st_mtime,
        )
        for _stale in _existing[:-PRERELOAD_SNAPSHOT_MAX_KEEP]:
            try:
                _stale.unlink()
            except OSError as _e:
                print(f'[WARN] could not remove stale prereload snapshot '
                      f'{_stale}: {_e}')
    if not _best_ckpt_path.exists():
        return resume_step
    ckpt = torch.load(_best_ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    try:
        optim.load_state_dict(ckpt['optimizer_state_dict'])
    except (ValueError, KeyError):
        pass
    s = ckpt.get('step', 0)
    p = ckpt.get('val_ppl', float('nan'))
    del ckpt
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    print(f'[watchdog] Reloaded best: step {s:,} PPL {p:.2f}')
    return s


steps_this_session = 0

# -- Schedule summary --
if LR_SCHEDULE == 'wsd':
    _warmup_end = int(WSD_WARMUP_FRAC * TOTAL_STEPS)
    _stable_end = int((WSD_WARMUP_FRAC + WSD_STABLE_FRAC) * TOTAL_STEPS)
    _sched_str = (f'WSD: warmup 0->{_warmup_end:,}, stable {_warmup_end:,}->{_stable_end:,}, '
                  f'decay {_stable_end:,}->{TOTAL_STEPS:,}, floor={WSD_LR_FLOOR:.2e}')
else:
    _sched_str = f'cosine: warmup {WARMUP_STEPS:,} steps'

print(f'\n{"="*60}')
print(f'CfC/BAOAB [{INTEGRATOR}]: steps {resume_step+1:,} -> {TOTAL_STEPS:,}')
print(f'  batch={BATCH_SIZE} x accum={GRAD_ACCUM} (eff={EFFECTIVE_BATCH})')
print(f'  block={BLOCK_SIZE}  lr={LR}  grad_clip={GRAD_CLIP}')
print(f'  schedule: {_sched_str}')
print(f'  d={model_cfg.d}  L={model_cfg.L}  M={model_cfg.n_registers}')
print(f'  fock-reg: lambda={LAMBDA_FOCK_REG}  eps={FOCK_REG_EPS}')
print(f'  integrator: {INTEGRATOR}  (cfg.integrator={model_cfg.integrator}, '
      f'analytic V_theta force={model_cfg.vtheta_analytic_force}, '
      f'thermostat T={model_cfg.langevin_T:g})')
print(f'  watchdog: threshold={GRAD_NORM_EMA_THRESHOLD} patience={GRAD_NORM_EMA_PATIENCE} '
      f'hard_trigger={GRAD_NORM_HARD_TRIGGER}')
print(f'  spike-capture: {"on" if CAPTURE_SPIKE_BATCH else "off"}'
      f'  capture_threshold={CAPTURE_SPIKE_THRESHOLD}'
      f'  keep={SPIKEBATCH_SNAPSHOT_MAX_KEEP}')
print(f'  per-group clip: default={GRAD_CLIP}  overrides={GRAD_CLIP_OVERRIDES}')
if REVERSE_CHANNEL:
    _rev_mode = ('stable (QK-norm + '
                 + ('soft-norm' if REVERSE_CHANNEL_SOFT_NORM else 'RMS-norm')
                 + (' + pre-LN' if REVERSE_CHANNEL_PRE_LN else '') + ')')
    print(f'  reverse channel: {_rev_mode}  warmup={REVERSE_CHANNEL_WARMUP_STEPS} forwards')
else:
    print('  reverse channel: OFF')
print(f'{"="*60}\n')


# 2026-08-30 (companion note SS37): assign_clip_group / per_group_grad_norms
# / clip_grads_per_group moved out to grad_clip_utils.py so Cell 6d's replay
# helpers can import them directly instead of silently depending on Cell 6
# having already run first (see that module's docstring for why).
from grad_clip_utils import GradClipConfig, per_group_grad_norms, clip_grads_per_group

_GRAD_CLIP_CFG = GradClipConfig(
    default_clip=GRAD_CLIP,
    overrides=GRAD_CLIP_OVERRIDES,
    watchdog_exclude_groups=frozenset(WATCHDOG_EXCLUDE_GROUPS),
)

_last_pg_norms = {}
_last_spike_step = -10**9


def run_training(start_step, total_steps):
    """Run training steps [start_step, total_steps).

    2026-08-30 (companion note SS37): this used to be a bare top-level
    'for step in range(resume_step, TOTAL_STEPS):' loop. Interrupting it
    (Runtime -> Interrupt execution) to go inspect spike captures left the
    interrupted iteration's tensors (x, y, loss, ...) alive as ordinary
    notebook globals -- and, worse, kept the *entire* forward/backward
    graph for that half-finished step pinned in GPU memory, since nothing
    ever went out of scope. On this run that reliably OOM'd every
    replay_spike_batch() call afterward (78+ GiB 'allocated', not just
    reserved-but-cached) until a manual gc.collect()+empty_cache() dance
    cleared it. Resuming also required a full checkpoint-save -> restart
    runtime -> resume_ckpt override dance, because Cell 5 unconditionally
    rebuilds `model` from scratch and this cell's own resume block always
    reloads from whatever's on disk.

    As a real function, an interrupt here unwinds this function's stack
    frame (caught by the try/except around the call below) -- every
    per-step local (x, y, loss, xb, yb, ...) is freed within that same
    frame teardown, same as a normal return. To keep training after
    inspecting captures, just call this again with the step it reports:

        next_step = run_training(next_step, TOTAL_STEPS)

    No rebuild, no checkpoint reload, no runtime restart required -- those
    are now only needed when you actually want to pick up new *code*
    (e.g. a fresh git pull of this notebook), not just to look around.
    `save_manual_checkpoint(next_step)` below is still there for that case.
    """
    global run_ntp, run_vreg, run_fock_reg, n_run, n_skipped, best_val_ppl
    global _grad_norm_ema, _grad_norm_above_thresh, steps_this_session
    global _last_pg_norms, _last_spike_step, step

    for step in range(start_step, total_steps):
        lr_now = lr_schedule(step)
        for g in optim.param_groups:
            g['lr'] = lr_now

        optim.zero_grad(set_to_none=True)
        # Phase 1 spike-batch capture: snapshot the RNG state this step's
        # forward+backward is about to consume (the Langevin thermostat noise
        # draw depends on it) and start a fresh microbatch list. Cheap enough
        # to do unconditionally every step; discarded if this step is clean.
        _step_rng_cpu = torch.get_rng_state()
        _step_rng_cuda = torch.cuda.get_rng_state_all() if DEVICE == 'cuda' else None
        _this_step_batches = []
        accum_ntp = 0.0
        accum_vreg = 0.0
        accum_fock_reg = 0.0
        accum_rep = 0.0
        for _acc in range(GRAD_ACCUM):
            xb, yb = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, rng)
            _this_step_batches.append((xb.copy(), yb.copy()))
            x = torch.from_numpy(xb).to(DEVICE)
            y = torch.from_numpy(yb).to(DEVICE)
            loss, loss_ntp, v_reg, fock_reg = forward_with_vreg(
                x, y, LAMBDA_V, LAMBDA_FOCK_REG, FOCK_REG_EPS)
            if REGISTER_REPULSION:
                _rep = model.pop_repulsion_loss()
                loss = loss + _rep
                accum_rep += float(_rep.detach()) / GRAD_ACCUM
            (loss / GRAD_ACCUM).backward()
            accum_ntp      += loss_ntp.item()       / GRAD_ACCUM
            accum_vreg     += float(v_reg.detach()) / GRAD_ACCUM
            accum_fock_reg += float(fock_reg.detach()) / GRAD_ACCUM

        if GRAD_CENTRALIZATION:
            for p in model.parameters():
                if p.grad is not None and p.grad.dim() >= 2:
                    p.grad.sub_(p.grad.mean(dim=tuple(range(1, p.grad.dim())), keepdim=True))

        if PER_GROUP_CLIP:
            grad_norm, _last_pg_norms = clip_grads_per_group(model, _GRAD_CLIP_CFG)
        else:
            _last_pg_norms = per_group_grad_norms(model, _GRAD_CLIP_CFG) if GRAD_SPIKE_DEBUG else {}
            if model.V_phi is not None:
                nn.utils.clip_grad_norm_(model.V_phi.parameters(), GRAD_CLIP_VPHI)
            grad_norm = nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], GRAD_CLIP)

        if GRAD_SPIKE_DEBUG:
            _tot_preclip = float(grad_norm)
            if (_tot_preclip > GRAD_SPIKE_THRESHOLD
                    and (step - _last_spike_step) >= GRAD_SPIKE_COOLDOWN):
                _last_spike_step = step
                if _last_pg_norms:
                    _top = sorted(_last_pg_norms.items(),
                                  key=lambda kv: kv[1], reverse=True)[:8]
                    _brk = '  '.join(f'{k}={v:.1f}' for k, v in _top)
                else:
                    _brk = '(enable PER_GROUP_CLIP for breakdown)'
                print(f'\n[spike] step {step+1}: pre-clip total grad={_tot_preclip:.1f}  '
                      f'ntp={accum_ntp:.3f}  v_reg={accum_vreg:.4f}  fock_reg={accum_fock_reg:.4f}')
                print(f'[spike]   top groups: {_brk}')
                _log_write(json.dumps({
                    'step': step + 1, 'event': 'grad_spike',
                    'pre_clip_grad_norm': round(_tot_preclip, 2),
                    'ntp': round(accum_ntp, 4), 'v_reg': round(accum_vreg, 4),
                    'fock_reg': round(accum_fock_reg, 4),
                    'top_groups': {k: round(v, 2) for k, v in _top} if _last_pg_norms else {},
                }) + '\n')

        # Phase 1 spike-batch capture (companion note SS33.3, gate widened in
        # SS36 to CAPTURE_SPIKE_THRESHOLD -- deliberately independent of the
        # reload-triggering GRAD_NORM_HARD_TRIGGER below): if this step crosses
        # CAPTURE_SPIKE_THRESHOLD, snapshot NOW -- model.parameters() still hold
        # the values the forward+backward above actually saw, because
        # optim.step() (which mutates them) has not run yet. By the time the
        # watchdog block further down fires (if it fires at all -- most
        # captures under this lower threshold will NOT trip the hard reload),
        # it is too late: optim.step() has already applied the update, which
        # is why the existing '_prereload' snapshot alone cannot reproduce it.
        _raw_gn_pre = float(grad_norm)
        if (CAPTURE_SPIKE_BATCH and CAPTURE_SPIKE_THRESHOLD is not None
                and _raw_gn_pre > CAPTURE_SPIKE_THRESHOLD):
            _top_sb = (sorted(_last_pg_norms.items(), key=lambda kv: kv[1], reverse=True)[:8]
                       if _last_pg_norms else [])
            _spike_bundle = {
                'step': step + 1,
                'pre_clip_grad_norm': round(_raw_gn_pre, 2),
                'ntp': round(accum_ntp, 4), 'v_reg': round(accum_vreg, 4),
                'fock_reg': round(accum_fock_reg, 4),
                'top_groups': {k: round(v, 2) for k, v in _top_sb},
                'model_state_dict': {k: v.detach().cpu().clone()
                                      for k, v in model.state_dict().items()},
                'batches': _this_step_batches,
                'grad_accum': GRAD_ACCUM,
                'rng_state_cpu': _step_rng_cpu,
                'rng_state_cuda': _step_rng_cuda,
            }
            _sb_path = CKPT_DIR / f'{CKPT_PREFIX}_step{step+1}_spikebatch.pt'
            torch.save(_spike_bundle, _sb_path)
            _existing_sb = sorted(
                CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_spikebatch.pt'),
                key=lambda p: p.stat().st_mtime,
            )
            for _stale_sb in _existing_sb[:-SPIKEBATCH_SNAPSHOT_MAX_KEEP]:
                try:
                    _stale_sb.unlink()
                except OSError as _e:
                    print(f'[WARN] could not remove stale spikebatch snapshot '
                          f'{_stale_sb}: {_e}')
            print(f'[spike-capture] saved pre-step (weights, '
                  f'{len(_this_step_batches)} microbatch(es), rng) -> {_sb_path.name}')

        if torch.isfinite(grad_norm) and math.isfinite(accum_ntp):
            optim.step()
            for bank in model.V_theta.banks:
                if hasattr(bank, 'clamp_params'):
                    bank.clamp_params()
        else:
            n_skipped += 1
            optim.zero_grad(set_to_none=True)

        # -- Watchdog --
        _raw_gn = float(grad_norm)
        _grad_norm_ema = (1 - GRAD_NORM_EMA_ALPHA) * _grad_norm_ema + GRAD_NORM_EMA_ALPHA * _raw_gn
        if _grad_norm_ema > GRAD_NORM_EMA_THRESHOLD:
            _grad_norm_above_thresh += 1
        else:
            _grad_norm_above_thresh = 0

        if _grad_norm_above_thresh >= GRAD_NORM_EMA_PATIENCE:
            print(f'\n[watchdog] EMA grad_norm={_grad_norm_ema:.1f} > {GRAD_NORM_EMA_THRESHOLD} '
                  f'for {_grad_norm_above_thresh} steps at step {step+1}.')
            if _last_pg_norms:
                _top = sorted(_last_pg_norms.items(), key=lambda kv: kv[1], reverse=True)[:5]
                print('[watchdog] top group norms (pre-clip): '
                      + ', '.join(f'{k}={v:.1f}' for k, v in _top))
            _log_write(json.dumps({
                'step': step + 1, 'event': 'watchdog_reload',
                'ema_grad_norm': round(_grad_norm_ema, 2),
                'above_thresh_steps': _grad_norm_above_thresh,
            }) + '\n')
            _reload_best(step + 1)
            _grad_norm_ema = 0.0
            _grad_norm_above_thresh = 0
            n_skipped += 1
        elif GRAD_NORM_HARD_TRIGGER is not None and _raw_gn > GRAD_NORM_HARD_TRIGGER:
            # Fast path: a single step this far above threshold (e.g. the
            # 2401.6 / 1815.9 spikes seen 2026-08-23) can already do lasting
            # damage before the slow EMA (alpha=0.05, patience=200 consecutive
            # steps) would ever fire. Reload immediately, no smoothing.
            print(f'\n[watchdog-hard] single-step grad_norm={_raw_gn:.1f} > '
                  f'{GRAD_NORM_HARD_TRIGGER} at step {step+1}.')
            if _last_pg_norms:
                _top = sorted(_last_pg_norms.items(), key=lambda kv: kv[1], reverse=True)[:5]
                print('[watchdog-hard] top group norms (pre-clip): '
                      + ', '.join(f'{k}={v:.1f}' for k, v in _top))
            _log_write(json.dumps({
                'step': step + 1, 'event': 'watchdog_hard_reload',
                'raw_grad_norm': round(_raw_gn, 2),
                'top_groups': {k: round(v, 2) for k, v in _top} if _last_pg_norms else {},
            }) + '\n')
            _reload_best(step + 1)
            _grad_norm_ema = 0.0
            _grad_norm_above_thresh = 0
            n_skipped += 1

        run_ntp += accum_ntp
        run_vreg += accum_vreg
        run_fock_reg += accum_fock_reg
        n_run += 1
        steps_this_session += 1

        if (step + 1) % LOG_INTERVAL == 0:
            avg_ntp = run_ntp / n_run
            avg_vreg = run_vreg / n_run
            avg_fock_reg = run_fock_reg / n_run
            run_ntp, run_vreg, run_fock_reg, n_run = 0.0, 0.0, 0.0, 0
            elapsed = time.time() - t0
            sec_per_step = elapsed / steps_this_session
            remaining = (TOTAL_STEPS - step - 1) * sec_per_step
            alphas = model.xi_alpha_values()
            alpha_str = ','.join(f'{a:.3f}' for a in alphas)
            _top_grp = ''
            _dc_ratio = None
            if PER_GROUP_CLIP and _last_pg_norms:
                _k, _v = max(_last_pg_norms.items(), key=lambda kv: kv[1])
                _top_grp = f'top[{_k}]={_v:.1f}  '
                # dc_ratio (companion note SS38.7): depth_code's own group
                # norm vs. the next-largest group. Discovered post hoc from
                # the 7 captured spike events -- every smooth-cascade one had
                # dc_ratio < 1.8, both localized-blowup ones had > 2.2 -- but
                # that was only ever checked AT capture time (7 points total).
                # Logging it on every LOG_INTERVAL step here (cheap: the
                # breakdown is already computed above for clipping) is what
                # lets a future mining pass check whether it's actually a
                # leading indicator during ordinary training, not just a
                # spike-time coincidence.
                _dc_norm = _last_pg_norms.get('override:depth_code')
                if _dc_norm is not None:
                    _others = [v for k, v in _last_pg_norms.items()
                               if k != 'override:depth_code']
                    _second = max(_others) if _others else 0.0
                    _dc_ratio = _dc_norm / _second if _second > 0 else float('inf')
                    _top_grp += f'dc_ratio={_dc_ratio:.2f}  '

            # weight-space stiffness proxy (companion note SS39.4/SS39.5):
            # SS39's per-row attribution and V_theta exponent-occupancy tests
            # both failed to discriminate the localized-mode spike captures
            # from the smooth-cascade ones -- the localized events were, if
            # anything, the *flatter* ones across rows, and occupancy showed
            # no separation at all. That argues the discriminator (if it
            # exists) lives in the WEIGHTS feeding the low-rank precision
            # factor, not in any particular batch. sigma_max(B_proj.weight)
            # bounds how large ||B_proj(xi)||_F can get for ANY unit-norm xi,
            # *before* _bound_lowrank's runtime tanh cap (SS28/model_aniso_
            # gaussian_vtheta.py) ever engages -- unlike everything tried in
            # SS39, it is a pure function of the current weights, with zero
            # dependence on which tokens are in this step's batch. An SVD of
            # a (K*d*rank, in_d) matrix is cheap relative to a training step
            # but is real compute (unlike dc_ratio, which just reads numbers
            # already computed for clipping), so -- same as dc_ratio -- this
            # only runs at LOG_INTERVAL cadence, not every step.
            _bproj_sigma_max = None
            _bproj_sigma_by_bank = None
            try:
                _vt_banks = model.V_theta.bank.banks
                _sigmas = []
                with torch.no_grad():
                    for _bk in _vt_banks:
                        _bp = getattr(_bk, 'B_proj', None)
                        if _bp is None:
                            continue
                        _sigmas.append(float(torch.linalg.matrix_norm(
                            _bp.weight.detach(), ord=2)))
                if _sigmas:
                    _bproj_sigma_by_bank = [round(s, 4) for s in _sigmas]
                    _bproj_sigma_max = max(_sigmas)
                    _top_grp += f'bproj_sig={_bproj_sigma_max:.2f}  '
            except Exception as _e:
                # best-effort diagnostic (e.g. a non-aniso V_theta variant
                # without .bank.banks) -- must never break the training loop.
                if GRAD_SPIKE_DEBUG:
                    print(f'[warn] b_proj sigma_max diagnostic failed: {_e}')

            _rep_str = f'rep={accum_rep:.4f}  ' if REGISTER_REPULSION else ''
            _mem_alloc_gb = torch.cuda.memory_allocated() / 1e9
            _mem_resv_gb = torch.cuda.memory_reserved() / 1e9
            _mem_peak_gb = torch.cuda.max_memory_allocated() / 1e9
            torch.cuda.reset_peak_memory_stats()
            print(
                f'step {step+1:7d}/{TOTAL_STEPS}  '
                f'ntp={avg_ntp:.4f}  v_reg={avg_vreg:.4f}  fock_reg={avg_fock_reg:.4f}  '
                f'lr={lr_now:.2e}  grad={float(grad_norm):.2f}  {_rep_str}{_top_grp}'
                f'gamma={model.gamma.item():.3f}  alpha=[{alpha_str}]  '
                f'mem_alloc={_mem_alloc_gb:.1f}GB  mem_resv={_mem_resv_gb:.1f}GB  '
                f'mem_peak={_mem_peak_gb:.1f}GB  '
                f'{elapsed:.0f}s  (~{remaining/3600:.1f}h remaining)')
            _log_write(json.dumps({
                'step': step + 1, 'train_loss': avg_ntp, 'v_reg': avg_vreg,
                'fock_reg': avg_fock_reg,
                'lr': lr_now, 'grad_norm': float(grad_norm),
                'gamma': model.gamma.item(), 'xi_alphas': alphas,
                'reg_repulsion': accum_rep,
                'dc_ratio': (round(_dc_ratio, 4)
                             if _dc_ratio is not None and math.isfinite(_dc_ratio)
                             else _dc_ratio),
                'b_proj_sigma_max': (round(_bproj_sigma_max, 4)
                                     if _bproj_sigma_max is not None else None),
                'b_proj_sigma_by_bank': _bproj_sigma_by_bank,
                'mem_alloc_gb': round(_mem_alloc_gb, 3),
                'mem_reserved_gb': round(_mem_resv_gb, 3),
                'mem_peak_gb': round(_mem_peak_gb, 3),
                'elapsed_sec': elapsed, 'sec_per_step': sec_per_step,
            }) + '\n')

        if (step + 1) % EVAL_INTERVAL == 0:
            val_loss = evaluate()
            val_ppl = math.exp(val_loss)
            is_best = val_ppl < best_val_ppl
            if is_best:
                best_val_ppl = val_ppl
            elapsed = time.time() - t0
            marker = '*** NEW BEST ***' if is_best else ''
            print(f'>>> EVAL step {step+1:,}  val_loss={val_loss:.4f}  '
                  f'val_ppl={val_ppl:.2f}  best={best_val_ppl:.2f}  '
                  f'{marker}  ({elapsed:.0f}s)')
            _log_write(json.dumps({
                'step': step + 1, 'val_loss': val_loss,
                'val_ppl': val_ppl, 'best_ppl': best_val_ppl,
            }) + '\n')
            if is_best:
                save_checkpoint(step + 1, val_loss, tag_suffix='_best')

        if (step + 1) in set(CKPT_STEPS):
            if (step + 1) % EVAL_INTERVAL != 0:
                val_loss = evaluate()
                val_ppl = math.exp(val_loss)
            save_checkpoint(step + 1, val_loss)

        if PROBE_MAX_STEPS is not None and (step + 1) >= PROBE_MAX_STEPS:
            if (step + 1) % EVAL_INTERVAL != 0:
                val_loss = evaluate()
                val_ppl = math.exp(val_loss)
            save_checkpoint(step + 1, val_loss, tag_suffix='_probe_stop')
            print(f'\n[probe] PROBE_MAX_STEPS={PROBE_MAX_STEPS} reached at step '
                  f'{step+1:,}; stopping this side-by-side probe (val_ppl='
                  f'{val_ppl:.2f}). Saved {CKPT_PREFIX}_step{step+1}_probe_stop.pt')
            _log_write(json.dumps({
                'step': step + 1, 'event': 'probe_max_steps_stop',
                'val_ppl': val_ppl,
            }) + '\n')
            break

        if CAUSAL_PROBE_INTERVAL > 0 and (step + 1) % CAUSAL_PROBE_INTERVAL == 0:
            _cp_passed, _cp_delta = run_causal_probe(step + 1)
            _log_write(json.dumps({
                'step': step + 1,
                'causal_probe_passed': _cp_passed,
                'causal_probe_max_delta': _cp_delta,
            }) + '\n')

        if TRAINED_LEAK_PROBE_INTERVAL > 0 and (step + 1) % TRAINED_LEAK_PROBE_INTERVAL == 0:
            _tlp_result = run_trained_leak_probe(step + 1)
            _log_write(json.dumps(_tlp_result) + '\n')

    _log_fh[0].close()
    print(f'\nTraining complete. Best PPL: {best_val_ppl:.2f}')
    return total_steps


def save_manual_checkpoint(next_step_num, val_loss_placeholder=4.6):
    """Force-save current in-memory model/optimizer state under a
    distinct '_manual' filename, without waiting for the next scheduled or
    best checkpoint. `next_step_num` should be whatever run_training() most
    recently returned (i.e. the step training will resume from).

    This is only needed before an *actual* runtime restart (e.g. to pick
    up freshly-pulled notebook code) -- interrupting to inspect captures
    and calling run_training(next_step, TOTAL_STEPS) again no longer needs
    it (companion note SS37).
    """
    return save_checkpoint(next_step_num, val_loss_placeholder, tag_suffix='_manual')


try:
    next_step = run_training(resume_step, TOTAL_STEPS)
except KeyboardInterrupt:
    print(f'\n[run_training] interrupted at step {step + 1:,}. Call '
          f'run_training({step + 1}, TOTAL_STEPS) to resume in-place '
          f'(no rebuild / no checkpoint reload / no runtime restart needed).')
    next_step = step + 1
else:
    print(f'\n[run_training] finished cleanly; next_step={next_step} '
          f'(== TOTAL_STEPS means training is fully done).')

In [ ]:
# == Cell 6b: Stiffness diagnostic — how much is the CfC actually saving? ==
#
# The explicit (Verlet) layer step is stable only while omega*dt < 2, where
# omega = sqrt(K/m) is the local V_theta curvature seen by one coordinate of
# one token.  This probe measures the distribution of omega*dt across a real
# batch, so the instability can be observed *directly* rather than inferred
# from the gradient norm after the fact.
#
# Read it as: any mass above omega*dt = 2 is a coordinate the Verlet
# integrator is provably amplifying, and that the CfC propagator rotates
# instead.  Safe to run against any arm -- it temporarily borrows the
# harmonic linearisation even when training under 'verlet'.

import contextlib

def stiffness_report(mdl, x, dt=None):
    """Distribution of omega*dt over layers, tokens and dimensions."""
    dt = float(mdl.cfg.dt if dt is None else dt)
    if not hasattr(mdl.V_theta, 'harmonic_terms'):
        raise RuntimeError('V_theta has no harmonic_terms(); need the '
                           'anisotropic Gaussian family.')

    seen = []
    _orig = mdl.V_theta.harmonic_terms

    def _recording(xis, h):
        k_diag, s = _orig(xis, h)
        seen.append(k_diag.detach().float().flatten().cpu())
        return k_diag, s

    _saved = (mdl.cfg.integrator, mdl.cfg.vtheta_analytic_force)
    mdl.V_theta.harmonic_terms = _recording
    mdl.cfg.integrator, mdl.cfg.vtheta_analytic_force = 'baoab_cfc', True
    try:
        was_training = mdl.training
        mdl.eval()
        with torch.enable_grad():
            mdl(x)
    finally:
        mdl.V_theta.harmonic_terms = _orig
        mdl.cfg.integrator, mdl.cfg.vtheta_analytic_force = _saved
        if was_training:
            mdl.train()

    k = torch.cat(seen)
    m = float(mdl.compute_mass(x).mean())
    wdt = (k.clamp(min=0) / m).sqrt() * dt
    n_total = int(wdt.numel())
    # The tails are what matter, so keep the exact max but subsample for the
    # quantiles: torch.quantile refuses inputs beyond ~16M elements, and
    # L*B*T*d reaches that at d=384, L=16 with a large auto-probed batch.
    wdt_max = float(wdt.max())
    if n_total > 4_000_000:
        idx = torch.randint(0, n_total, (4_000_000,))
        wdt_q = wdt[idx]
    else:
        wdt_q = wdt
    q = torch.tensor([0.5, 0.9, 0.99, 0.999])
    qs = torch.quantile(wdt_q.double(), q.double()).float()
    qs = torch.cat([qs, torch.tensor([wdt_max])])
    return {
        'n_samples': n_total,
        'mean_mass': m,
        'median': float(qs[0]), 'p90': float(qs[1]), 'p99': float(qs[2]),
        'p999': float(qs[3]), 'max': float(qs[4]),
        'frac_unstable': float((wdt > 2.0).float().mean()),
        'frac_marginal': float((wdt > 1.0).float().mean()),
    }


_rng_s = np.random.default_rng(0)
_xb, _ = get_batch(train_ids, min(BATCH_SIZE, 4), BLOCK_SIZE, _rng_s)
_rep = stiffness_report(model, torch.from_numpy(_xb).to(DEVICE))

print(f'omega*dt over {_rep["n_samples"]:,} (layer, token, dim) samples '
      f'  [mean mass {_rep["mean_mass"]:.3f}]')
print(f'  median {_rep["median"]:.4f}   p90 {_rep["p90"]:.4f}   '
      f'p99 {_rep["p99"]:.4f}   p99.9 {_rep["p999"]:.4f}   '
      f'max {_rep["max"]:.4f}')
print(f'  fraction with omega*dt > 1 (marginal): {_rep["frac_marginal"]:.3e}')
print(f'  fraction with omega*dt > 2 (Verlet-unstable): '
      f'{_rep["frac_unstable"]:.3e}')
if _rep['max'] > 2.0:
    print('  => the explicit step is UNSTABLE on some coordinates right now; '
          'these are exactly what baoab_cfc integrates exactly instead.')
else:
    print('  => no coordinate exceeds the explicit stability bound at this '
          'checkpoint (re-run later in training: wells sharpen over time).')


In [ ]:
# == Cell 6b-2: sigma_max(B_k)^2 diagnostic -- for bracketing PRECISION_LR_MAX ==
#
# stiffness_report (Cell 6b) reports omega*dt from k_diag, the DIAGONAL
# harmonic model -- exactly what baoab_cfc already integrates exactly, so
# it is immune to unbounded B_k growth by construction (SS24, companion
# note).  The channel that is NOT immune is the anisotropic low-rank
# correction B_k B_k^T, which baoab_cfc still demotes to an explicit kick
# (SS28.2/SS28.6) and which PRECISION_LR_MAX (SS29.3) / baoab_cfc_lowrank
# (SS29.2) exist to fix.  This cell measures that raw quantity directly --
# sigma_max(B_k)^2, per well, per xi-channel, per layer, BEFORE it is
# mixed with a_k or gated by the Gaussian bump g_k -- so a
# PRECISION_LR_MAX budget can be bracketed (SS31.4) between this
# checkpoint's distribution and a spike-regime checkpoint's, instead of
# guessed.  Mirrors the sigma_lr_* percentiles added to SCAF's
# StiffnessProbe (semsimula-scaf, SS31.3).
#
# Usage: load the checkpoint of interest into `model` (the existing
# reload/checkpoint-loading cell already does this), then re-run this
# cell.  Comparing a healthy checkpoint (e.g. this run's step-27,000 best)
# against a spike-regime one (the `_prereload` snapshots saved at the
# hard-trigger steps) is the SS31.2 bracketing protocol.

def sigma_lr_report(mdl, x):
    """Distribution of the raw sigma_max(B_k)^2 over layers, wells and
    xi-channels -- the quantity PRECISION_LR_MAX caps directly. Deliberately
    NOT combined with a_k or g_k (unlike stiffness_report's omega*dt):
    bracketing a PRECISION_LR_MAX budget needs the raw per-well spectral
    norm, not a quantity already mixed with the bump weight or the
    diagonal precision.
    """
    if not hasattr(mdl.V_theta, 'context_components'):
        raise RuntimeError('V_theta has no context_components(); need the '
                           'anisotropic Gaussian family.')

    seen = []
    _orig = mdl.V_theta.context_components

    def _recording(xis):
        comps = _orig(xis)
        # comps: list of (mu, a, w, B) tuples, one per xi-channel (see
        # AnisotropicMultiContextGaussianVTheta.context_components).
        for (_mu, _a, _w, B) in comps:
            if B.shape[-1] == 0:
                continue
            # sigma_max(B_k)^2 = (largest singular value of B_k)^2, taken
            # from the SVD of B_k directly rather than eigvalsh(B_k^T B_k):
            # forming the Gram squares the condition number and can make
            # the symmetric-eigen driver fail to converge on degenerate
            # wells (same failure mode fixed in cfc_baoab.lowrank_modes).
            sigma_max_sq = torch.linalg.svdvals(B)[..., 0] ** 2  # (..., K)
            seen.append(sigma_max_sq.detach().float().flatten().cpu())
        return comps

    mdl.V_theta.context_components = _recording
    try:
        was_training = mdl.training
        mdl.eval()
        with torch.enable_grad():
            mdl(x)
    finally:
        mdl.V_theta.context_components = _orig
        if was_training:
            mdl.train()

    if not seen:
        return {'n_samples': 0}
    s = torch.cat(seen)
    n_total = int(s.numel())
    s_max = float(s.max())
    if n_total > 4_000_000:
        idx = torch.randint(0, n_total, (4_000_000,))
        s_q = s[idx]
    else:
        s_q = s
    q = torch.tensor([0.5, 0.9, 0.99, 0.999])
    qs = torch.quantile(s_q.double(), q.double()).float()
    qs = torch.cat([qs, torch.tensor([s_max])])
    return {
        'n_samples': n_total,
        'p50': float(qs[0]), 'p90': float(qs[1]), 'p99': float(qs[2]),
        'p999': float(qs[3]), 'max': float(qs[4]),
    }


_rng_slr = np.random.default_rng(0)
_xb_slr, _ = get_batch(train_ids, min(BATCH_SIZE, 4), BLOCK_SIZE, _rng_slr)
_rep_slr = sigma_lr_report(model, torch.from_numpy(_xb_slr).to(DEVICE))

if _rep_slr['n_samples'] == 0:
    print('sigma_lr_report: no low-rank wells found (rank=0 or no '
          'context_components on this V_theta).')
else:
    print(f'sigma_max(B_k)^2 over {_rep_slr["n_samples"]:,} '
          f'(layer, xi-channel, well) samples:')
    print(f'  p50 {_rep_slr["p50"]:.3f}   p90 {_rep_slr["p90"]:.3f}   '
          f'p99 {_rep_slr["p99"]:.3f}   p99.9 {_rep_slr["p999"]:.3f}   '
          f'max {_rep_slr["max"]:.3f}')
    print('  => record this alongside the checkpoint step; compare a '
          'healthy checkpoint against a `_prereload` (spike-regime) '
          'snapshot to bracket PRECISION_LR_MAX per companion-note SS31.4 '
          '(set it above the healthy p95-p99, below the spike-regime '
          'p99/max, then bias slightly upward since the Frobenius cap is '
          'conservative by up to a factor ANISO_RANK).')


In [ ]:
# == Cell 6b-3: bracket PRECISION_LR_MAX -- healthy vs the captured spike
# bundles (companion note SS41.7 item 1 / SS31.2-SS31.4 protocol) ==
#
# sigma_lr_report (Cell 6b-2) needs `model` to already hold the weights of
# interest. Rather than trust "whatever happens to be loaded right now" for
# the spike-regime reading, this loads each `_spikebatch.pt` bundle's own
# `model_state_dict` -- bit-identical to what replay_precision_cap_ablation /
# replay_integrator_ablation (Cell 6d) actually replayed -- and the healthy
# `_best.pt` checkpoint, one at a time, restoring the live model's weights
# in a `finally` block. Same non-pollution invariant as the Cell 6d replay
# helpers, so this is safe to run against the live, still-resumable
# training state.
#
# Requires Cell 6b-2 to have run at least once (defines sigma_lr_report).

def bracket_precision_lr_max(step_tags=(47116, 48507, 48917), mdl=None,
                              healthy_ckpt_name=None, n_batch=4):
    mdl = mdl if mdl is not None else model
    _saved_sd = _copy.deepcopy(mdl.state_dict())
    _rng_b = np.random.default_rng(0)
    _xb_b, _ = get_batch(train_ids, min(BATCH_SIZE, n_batch), BLOCK_SIZE, _rng_b)
    _x_b = torch.from_numpy(_xb_b).to(DEVICE)

    reports = {}
    try:
        # -- healthy baseline --
        _healthy_path = (CKPT_DIR / healthy_ckpt_name if healthy_ckpt_name
                          else CKPT_DIR / f'{CKPT_PREFIX}_best.pt')
        _hd = torch.load(_healthy_path, map_location='cpu', weights_only=False)
        mdl.load_state_dict(
            {k: v.to(DEVICE) for k, v in _hd['model_state_dict'].items()},
            strict=False)
        reports[f'healthy (step {_hd.get("step", "?"):,})'] = sigma_lr_report(mdl, _x_b)
        del _hd

        # -- each captured spike bundle --
        for step_tag in step_tags:
            _path = CKPT_DIR / f'{CKPT_PREFIX}_step{step_tag}_spikebatch.pt'
            _bd = torch.load(_path, map_location='cpu', weights_only=False)
            mdl.load_state_dict(
                {k: v.to(DEVICE) for k, v in _bd['model_state_dict'].items()},
                strict=False)
            reports[f'spike step {step_tag}'] = sigma_lr_report(mdl, _x_b)
            del _bd
    finally:
        mdl.load_state_dict(_saved_sd)

    print(f'{"state":<22} {"p50":>9} {"p90":>9} {"p99":>9} {"p99.9":>9} {"max":>12}')
    for label, r in reports.items():
        print(f'{label:<22} {r["p50"]:9.3f} {r["p90"]:9.3f} {r["p99"]:9.3f} '
              f'{r["p999"]:9.3f} {r["max"]:12.3f}')
    print('\n=> per companion-note SS31.4: set PRECISION_LR_MAX above the '
          'healthy row\'s p90-p99, below the spike rows\' p99/max, then bias '
          f'slightly upward (the Frobenius cap in _bound_lowrank is '
          f'conservative by up to a factor ANISO_RANK={ANISO_RANK}).')
    return reports


_brackets = bracket_precision_lr_max()


In [ ]:
# == Cell 6c: Bottleneck profile — GPU-compute vs CPU/launch ==
#
# Interrupt the training cell first.  This does not step the optimizer
# and does not change weights.  Takes ~1–2 minutes on the live A100.
#
# Three independent measurements vote on why the step is ~22s:
#   (1) GPU-compute-bound  — H100 extra FLOPs/bandwidth can help
#   (2) CPU/launch-bound   — a faster GPU buys almost nothing
#
# Prefers scaleup/debug/cfc_step_bottleneck_profile.py when present
# (pull the paper repo).  Falls back to the copy shipped in this cell
# so a stale Colab clone still works.

import importlib.util
import sys
from pathlib import Path

def _load_bottleneck_mod():
    candidates = []
    if 'REPO_ROOT' in globals():
        candidates.append(
            REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup'
            / 'debug' / 'cfc_step_bottleneck_profile.py')
    candidates += [
        Path.cwd() / 'debug' / 'cfc_step_bottleneck_profile.py',
        Path.cwd() / 'scaleup' / 'debug' / 'cfc_step_bottleneck_profile.py',
    ]
    for p in candidates:
        if p.exists():
            spec = importlib.util.spec_from_file_location(
                'cfc_step_bottleneck_profile', p)
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            print(f'[bottleneck] loaded {p}')
            return mod
    return None

_bp = _load_bottleneck_mod()
if _bp is None:
    print('[bottleneck] debug module not on disk; using the copy in this cell')
    # --- embedded copy of cfc_step_bottleneck_profile.py (keep in sync) ---
    import math, os, re, subprocess, threading, time
    from typing import Callable, Iterable

    def _median(xs):
        ys = sorted(xs)
        n = len(ys)
        if n == 0:
            return float('nan')
        if n % 2:
            return ys[n // 2]
        return 0.5 * (ys[n // 2 - 1] + ys[n // 2])

    def _sync():
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    def _time_cuda(fn, n_warmup=1, n_repeat=3):
        for _ in range(n_warmup):
            fn(); _sync()
        samples = []
        for _ in range(n_repeat):
            _sync()
            t0 = time.perf_counter()
            fn(); _sync()
            samples.append(time.perf_counter() - t0)
        return _median(samples), samples

    def _classify_scale(ratio_hi_over_lo, hi, lo):
        expected = hi / lo
        slope = ((ratio_hi_over_lo - 1.0) / (expected - 1.0)
                 if expected > 1 else 0.0)
        if slope >= 0.70:
            return 'compute', slope
        if slope <= 0.25:
            return 'launch', slope
        return 'mixed', slope

    def _classify_overlap(efficiency):
        if efficiency <= 0.25:
            return 'compute'
        if efficiency >= 0.65:
            return 'launch'
        return 'mixed'

    def _classify_util(mean_util):
        if mean_util >= 70:
            return 'compute'
        if mean_util <= 35:
            return 'launch'
        return 'mixed'

    def _poll_smi(samples, stop_evt, interval=0.05):
        while not stop_evt.is_set():
            try:
                out = subprocess.check_output(
                    ['nvidia-smi',
                     '--query-gpu=utilization.gpu,utilization.memory',
                     '--format=csv,noheader,nounits'],
                    text=True, timeout=2,
                ).strip().splitlines()[0]
                gpu_u, mem_u = [float(x.strip()) for x in out.split(',')[:2]]
                samples.append((gpu_u, mem_u))
            except Exception:
                return
            stop_evt.wait(interval)

    def _parse_self_cuda_seconds(table):
        m = re.search(
            r'Self (?:CUDA|device) time total:\s*([0-9.]+)\s*(us|ms|s)',
            table, flags=re.I)
        if not m:
            return None
        val = float(m.group(1))
        return val * {'us': 1e-6, 'ms': 1e-3, 's': 1.0}[m.group(2).lower()]

    def _calibrate_gemm_count(device, target_s, dim=4096):
        a = torch.randn(dim, dim, device=device, dtype=torch.float32)
        b = torch.randn(dim, dim, device=device, dtype=torch.float32)
        _sync()
        t0 = time.perf_counter()
        for _ in range(4):
            _ = a @ b
        _sync()
        per = max((time.perf_counter() - t0) / 4.0, 1e-4)
        n = min(max(4, int(math.ceil(target_s / per))), 400)
        return a, b, n, per

    def _enqueue_gemms(a, b, n, stream):
        with torch.cuda.stream(stream):
            acc = a
            for _ in range(n):
                acc = acc @ b
            stream_result = acc.sum()
        return stream_result

    def run_bottleneck_profile(
        model, *, forward_fn, make_batch, batch_size, grad_accum,
        batch_sizes=None, n_warmup=1, n_repeat=3, device='cuda',
        profile_kernels=True,
    ):
        if not torch.cuda.is_available():
            raise RuntimeError('Needs CUDA; CPU cannot split compute vs launch.')
        model.train()
        sizes = list(batch_sizes) if batch_sizes is not None else [1, 2, batch_size]
        sizes = sorted({s for s in sizes if 1 <= s <= batch_size})
        if batch_size not in sizes:
            sizes.append(batch_size); sizes.sort()
        cached = {}
        for bs in sizes:
            x, y = make_batch(bs)
            cached[bs] = (x.detach().clone(), y.detach().clone())

        def microbatch(bs, zero=True):
            x, y = cached[bs]
            loss = forward_fn(x, y)
            loss.backward()
            if zero:
                model.zero_grad(set_to_none=True)
            return loss

        print('=' * 64)
        print('CfC step bottleneck profile')
        cfg = getattr(model, 'cfg', None)
        print(f'  integrator={getattr(cfg, "integrator", "?")}  '
              f'd={getattr(cfg, "d", "?")}  L={getattr(cfg, "L", "?")}  '
              f'bs={batch_size}  accum={grad_accum}')
        print('  interrupt training first; this does not step the optimizer')
        print('=' * 64)
        votes = []

        print('\n[1] Batch scaling (one microbatch, fwd+bwd)')
        scale = {}
        for bs in sizes:
            med, samples = _time_cuda(
                lambda bs=bs: microbatch(bs), n_warmup=n_warmup, n_repeat=n_repeat)
            scale[bs] = med
            extra = (f'   ({bs}/{sizes[0]} = {med / scale[sizes[0]]:.2f})'
                     if bs != sizes[0] else '')
            samp = ' '.join(f'{s:.2f}' for s in samples)
            print(f'  bs={bs:<3d}  median {med:6.2f}s  samples [{samp}]{extra}')
        lo, hi = sizes[0], sizes[-1]
        ratio = scale[hi] / scale[lo] if scale[lo] > 0 else float('inf')
        scale_cls, slope = _classify_scale(ratio, hi, lo)
        votes.append(scale_cls)
        print(f'  linear slope captured: {slope:.2f}  '
              f'(1.0 = time ∝ batch,  0.0 = time flat)')
        print(f'  => {scale_cls.upper()}')

        print(f'\n[2] Side-stream GEMM overlap (bs={batch_size})')
        T_step = scale[batch_size]
        a, b, n_gemm, per = _calibrate_gemm_count(device, T_step)
        print(f'  calibrated {n_gemm} x {a.shape[0]} GEMMs  (~{per*1000:.1f} ms each)')

        def gemm_pile():
            side = torch.cuda.Stream()
            res = _enqueue_gemms(a, b, n_gemm, side)
            torch.cuda.current_stream().wait_stream(side)
            return res

        T_gemm, _ = _time_cuda(gemm_pile, n_warmup=0, n_repeat=2)
        print(f'  T_gemm alone  {T_gemm:.2f}s')

        def both():
            side = torch.cuda.Stream()
            res = _enqueue_gemms(a, b, n_gemm, side)
            microbatch(batch_size)
            torch.cuda.current_stream().wait_stream(side)
            return res

        T_both, _ = _time_cuda(both, n_warmup=0, n_repeat=2)
        overlap = (T_step + T_gemm - T_both) / min(T_step, T_gemm)
        overlap = max(0.0, min(1.2, overlap))
        overlap_cls = _classify_overlap(overlap)
        votes.append(overlap_cls)
        print(f'  T_step={T_step:.2f}s  T_gemm={T_gemm:.2f}s  T_both={T_both:.2f}s')
        print(f'  overlap efficiency = {overlap:.2f}  '
              f'(1.0 = full overlap / idle SMs,  0.0 = no spare SMs)')
        print(f'  => {overlap_cls.upper()}')
        del a, b

        print(f'\n[3] nvidia-smi during one full step ({grad_accum} microbatches)')
        smi, stop_evt = [], threading.Event()
        th = threading.Thread(target=_poll_smi, args=(smi, stop_evt), daemon=True)
        th.start(); time.sleep(0.15)

        def full_step():
            model.zero_grad(set_to_none=True)
            for _ in range(grad_accum):
                microbatch(batch_size, zero=False)
            model.zero_grad(set_to_none=True)

        _sync(); t0 = time.perf_counter(); full_step(); _sync()
        T_full = time.perf_counter() - t0
        stop_evt.set(); th.join(timeout=2.0)
        mean_u = None
        if smi:
            gpu_u = [g for g, _ in smi]
            mem_u = [m for _, m in smi]
            mean_u = sum(gpu_u) / len(gpu_u)
            p50_u = _median(gpu_u)
            p90_u = sorted(gpu_u)[max(0, int(0.9 * (len(gpu_u) - 1)))]
            util_cls = _classify_util(mean_u)
            votes.append(util_cls)
            print(f'  wall {T_full:.1f}s   samples={len(smi)}  '
                  f'GPU util mean {mean_u:.0f}%  p50 {p50_u:.0f}%  '
                  f'p90 {p90_u:.0f}%  mem-util mean {sum(mem_u)/len(mem_u):.0f}%')
            print(f'  => {util_cls.upper()}')
        else:
            print('  nvidia-smi unavailable; skipping this vote')
            print(f'  wall {T_full:.1f}s for the full step')

        if profile_kernels:
            print('\n[4] Profiler dump (one microbatch, supporting only)')
            try:
                from torch.profiler import ProfilerActivity, profile
                _sync(); t0 = time.perf_counter()
                with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
                             record_shapes=False, with_stack=False) as prof:
                    microbatch(batch_size); _sync()
                wall = time.perf_counter() - t0
                try:
                    table = prof.key_averages().table(
                        sort_by='self_cuda_time_total', row_limit=15)
                except Exception:
                    table = prof.key_averages().table(
                        sort_by='self_device_time_total', row_limit=15)
                self_cuda = _parse_self_cuda_seconds(table)
                if self_cuda is not None and wall > 0:
                    print(f'  wall {wall:.2f}s   Self-CUDA {self_cuda:.2f}s   '
                          f'busy={self_cuda / wall:.0%}')
                print(table)
            except Exception as exc:
                print(f'  profiler skipped: {type(exc).__name__}: {exc}')

        counts = {k: votes.count(k) for k in ('compute', 'launch', 'mixed')}
        if counts['compute'] > counts['launch'] and counts['compute'] >= 2:
            verdict = 'compute'
        elif counts['launch'] > counts['compute'] and counts['launch'] >= 2:
            verdict = 'launch'
        else:
            verdict = 'mixed'
        print('\n' + '=' * 64)
        if verdict == 'compute':
            print('VERDICT: (1) GPU-compute-bound')
            print('  The SMs are busy.  H100 extra FLOPs / HBM bandwidth can')
            print('  plausibly cut wall time by ~1.5-2.5x at the same 4x8.')
        elif verdict == 'launch':
            print('VERDICT: (2) CPU / launch-bound')
            print('  The GPU is idle between many small kernels.  A faster')
            print('  GPU buys almost nothing (maybe 1.0-1.2x).  The real')
            print('  levers are fewer sequential launches: cut GRAD_ACCUM,')
            print('  shrink ANISO_RANK, or fuse the per-layer Python step.')
        else:
            print('VERDICT: mixed — both (1) and (2) contribute')
            print('  H100 may help, but not by 2x.  Expect something closer')
            print('  to 1.2-1.6x unless launches are also reduced.')
        print(f'  votes: {votes}')
        print('=' * 64)
        model.zero_grad(set_to_none=True)
        return {'verdict': verdict, 'votes': votes}

    def run_from_notebook():
        import numpy as np
        rng = np.random.default_rng(12345)
        if not bool(globals().get('REGISTER_REPULSION', False)):
            def forward_fn(x, y):
                loss, *_ = forward_with_vreg(
                    x, y, LAMBDA_V, LAMBDA_FOCK_REG, FOCK_REG_EPS)
                return loss
        else:
            def forward_fn(x, y):
                loss, *_ = forward_with_vreg(
                    x, y, LAMBDA_V, LAMBDA_FOCK_REG, FOCK_REG_EPS)
                return loss + model.pop_repulsion_loss()

        def make_batch(bs):
            xb, yb = get_batch(train_ids, bs, BLOCK_SIZE, rng)
            return (torch.from_numpy(xb).to(DEVICE),
                    torch.from_numpy(yb).to(DEVICE))

        return run_bottleneck_profile(
            model, forward_fn=forward_fn, make_batch=make_batch,
            batch_size=int(BATCH_SIZE), grad_accum=int(GRAD_ACCUM),
            device=str(DEVICE))

    class _M: pass
    _bp = _M()
    _bp.run_from_notebook = run_from_notebook

_bp.run_from_notebook()


## Fock v2.1 component diagnosticsStandalone probe -- safe to run any time against the live model or afreshly loaded checkpoint. It answers two questions:1. **Structural health** -- is each Fock piece being *used well*?2. **PPL attribution** -- how much does each piece actually *buy*?

In [ ]:
# == Cell 7: Component diagnostics =====================================
import torch
_bp = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
_bd = torch.load(_bp, map_location=DEVICE, weights_only=False)
model.load_state_dict(_bd['model_state_dict'], strict=False)
model.eval()
print(f"Probe target -> {_bp.name}  step {_bd.get('step')}  PPL {_bd.get('val_ppl'):.2f}")
del _bd
import gc; gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()

import gc, math, sys, torch, numpy as np
for _a in ('last_traceback', 'last_value', 'last_type'):
    if hasattr(sys, _a): setattr(sys, _a, None)
model.zero_grad(set_to_none=True)
try: optim.zero_grad(set_to_none=True)
except Exception: pass
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
    _free, _total = torch.cuda.mem_get_info()
    print(f'GPU free {_free/1e9:.1f} / {_total/1e9:.1f} GB before probe')

PROBE_BS = 2
_rng = np.random.default_rng(1234)
def _mk(n, bs):
    return [(torch.from_numpy(a).to(DEVICE), torch.from_numpy(b).to(DEVICE))
            for a, b in (get_batch(val_ids, bs, BLOCK_SIZE, _rng) for _ in range(n))]
def _eval_on(batches):
    model.eval(); losses = []
    for x, y in batches:
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(float(loss.item()))
        del loss
    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    return float(np.mean(losses))

# --- 1. structural health ---
model.eval(); model.set_fock_capture(True)
_hx, _hy = _mk(1, PROBE_BS)[0]
with torch.enable_grad():
    _out = model(_hx, _hy)
del _out
rep = model.fock_component_report()
_cols = ['layer','active_frac','reg_cos_sim','create_entropy','create_alpha_max',
         'rev_entropy','rev_scale','qforce_ratio','destroy_mean']
print('='*72); print('Fock v2.1 STRUCTURAL HEALTH'); print('='*72)
print('  '.join(f'{c[:10]:>10}' for c in _cols))
for dd in rep['per_layer']:
    print('  '.join(f'{str(dd.get(c)):>10}' if isinstance(dd.get(c),(bool,type(None)))
                    else f'{float(dd.get(c)):>10.3f}' for c in _cols))
print('-'*72); print('summary:', {k: round(v,3) for k,v in rep['summary'].items()})
for f in rep.get('flags', []): print('  * '+f)
del _hx, _hy, rep; gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()

# --- 2. PPL attribution ---
_pb = _mk(40, PROBE_BS)
base = _eval_on(_pb); base_ppl = math.exp(base); rows = [('full model', base)]
if getattr(model, 'reverse_channel_scale', None) is not None:
    _s = model.reverse_channel_scale.detach().clone()
    with torch.no_grad(): model.reverse_channel_scale.zero_()
    rows.append(('  - reverse channel', _eval_on(_pb)))
    with torch.no_grad(): model.reverse_channel_scale.copy_(_s)
_thr = model.cfg.register_salience_threshold
try:
    model.cfg.register_salience_threshold = 1e9
    rows.append(('  - registers (all)', _eval_on(_pb)))
finally:
    model.cfg.register_salience_threshold = _thr
print('\n'+'='*72)
print(f"{'arm':<22}{'loss':>10}{'ppl':>10}{'dPPL':>10}")
for n, l in rows:
    p = math.exp(l); print(f'{n:<22}{l:>10.4f}{p:>10.2f}{p-base_ppl:>+10.2f}')

In [ ]:
# == Cell 8: Training curve ============================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

eval_entries = []
alpha_entries = []
if log_path.exists():
    with open(log_path) as f:
        for line in f:
            try:
                e = json.loads(line)
                if 'val_ppl' in e and 'event' not in e:
                    eval_entries.append(e)
                if 'xi_alphas' in e and 'event' not in e:
                    alpha_entries.append(e)
            except Exception:
                pass

if eval_entries:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    ax = axes[0]
    steps_arr = [e['step'] for e in eval_entries]
    ppls = [e['val_ppl'] for e in eval_entries]
    ax.plot(steps_arr, ppls, 'o-',
            label=f'Aniso-Gaussian r={ANISO_RANK} + fock-reg (OWT d=384)',
            linewidth=1.5, color='#C62828')
    ax.axhline(y=9.14, color='green', linestyle='--', alpha=0.7,
               label='Aniso-Gaussian TinyStories best (9.14)')
    ax.set_xlabel('Step')
    ax.set_ylabel('Val PPL')
    ax.set_title(f'Aniso-Gaussian + Fock-Reg -- OpenWebText d=384')
    ax.legend()
    ax.grid(True, alpha=0.3)

    if alpha_entries:
        ax = axes[1]
        a_steps = [e['step'] for e in alpha_entries]
        n_ch = len(alpha_entries[0]['xi_alphas'])
        for k in range(n_ch):
            ax.plot(a_steps, [e['xi_alphas'][k] for e in alpha_entries],
                    'o-', label=f'alpha_{k+1}', markersize=2, linewidth=1.5)
        ax.set_xlabel('Step')
        ax.set_ylabel('alpha_k')
        ax.set_title('Fock coupling strengths (alpha_k)')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_ylim(-0.05, 1.05)

    plt.tight_layout()
    fig.savefig(RESULTS_DIR / 'training_curve_aniso_gaussian_owt.png', dpi=150)
    plt.show()
    print(f'Saved: {RESULTS_DIR / "training_curve_aniso_gaussian_owt.png"}')
else:
    print('No eval data to plot.')